# Temporal Generalisation and Explanation Stability of Passive-Sensing Models for Depressive and Anxiety Symptoms

Analysis code for the manuscript of the same title. The notebook runs top to bottom (**Run all**) on the GLOBEM
dataset (version 1.1) and writes the figures, tables and key numbers reported in the manuscript.

## Data access

GLOBEM is distributed by PhysioNet under credentialed access (https://doi.org/10.13026/r9s1-s711). The data are
**not** included in this repository and may not be redistributed. After obtaining access, either

* place the PhysioNet zip (file name in `ZIP_NAME`, Section "Run configuration") in the folder given by
  `GLOBEM_DRIVE_ROOT`, or
* point `GLOBEM_DATA_ROOT` at an extracted copy containing the folders `INS-W_1` ... `INS-W_4`.

## Running

| Setting | Environment variable | Default |
|---|---|---|
| Folder holding the PhysioNet zip | `GLOBEM_DRIVE_ROOT` | Google Drive root on Colab; current directory otherwise |
| Extracted data | `GLOBEM_DATA_ROOT` | `<drive root>/GLOBEM_extracted` |
| Outputs and checkpoints | `GLOBEM_OUTPUT_ROOT` | `<drive root>/GLOBEM_JMIR_outputs` |
| Quick pipeline check with tiny budgets | `GLOBEM_SMOKE_TEST=1` | off |

The full configuration takes several hours on a GPU runtime with high RAM (sequences are held in GPU memory and the
correlation reducer runs on the GPU). Every expensive stage is checkpointed, so an interrupted run resumes where it
stopped. Each checkpoint is keyed by a fingerprint of the code, settings and upstream inputs it depends on, so
edited code never reloads stale results.

## Contents

| Section | Purpose | Main outputs |
|---|---|---|
| 1-4 | Load cohorts, clean features, build 28-day windows, self-deviation features, BDI-II table | Table 1 |
| 5 | Evaluation protocols: forward chaining (primary), leave-one-year-out, participant-disjoint | Figure 1 |
| 6-9 | Train-only preprocessing, models, Optuna tuning, ten-seed refitting | per-seed results |
| 10 | Two-level bootstrap, paired comparisons, Holm adjustment, equivalence | Table 2, Figure 3 |
| 11 | Calibration, prevalence scenarios, label-permutation control | Figure 4 |
| 12 | Collection-year shift: year classifiers, feature subsets, era transfer | Figure 2 |
| 13 | Explanation stability of XGBoost SHAP rankings | Figure 5 |
| 14 | Sensitivity analyses: 2018 excluded, end-of-term BDI-II, `_norm` ablation | supplementary |
| 15-17 | Figures, tables, key numbers, run manifest | all exported files |

## Design safeguards

* **Temporal ordering.** Forward chaining trains only on years before the test year. The 2021 forward split is
  identical to the 2021 leave-one-year-out split; it is trained once and reported under both protocols.
* **Train-only preprocessing.** The channel filter (drop channels with > 95% missing days) is fitted per split on the
  outer-training participants. Imputation, correlation reduction, variance filtering and scaling are fitted on the
  inner-training participants of each seed.
* **Model selection.** Hyperparameters, checkpoints and decision thresholds are selected on a participant-disjoint
  inner validation set. Outer test data are scored once per model, split and seed.
* **Comparable pooling.** Pooled estimates, comparison families and predictive values use only the test years
  available to every model (GroupDRO and IRM need two training years, so the first forward year is not pooled).
* **Uncertainty.** A two-level bootstrap resamples participants and draws a training seed per replicate; paired
  comparisons share participant resamples and are Holm-adjusted within family.
* **Strict outcome definition.** The BDI-II analysis stops with an error instead of falling back to another label.

The main analysis keeps GLOBEM's released `_norm` features; the comparison without them is run for the tabular
models and exported as supplementary material.

## Outputs and data protection

`main_figures/`, `main_tables/`, `supplementary/` and `key_numbers.json` contain aggregate results only.
`checkpoints/` holds per-window predictions with participant identifiers and must never be shared. Clear all cell
outputs before committing this notebook.

In [ ]:
# ============================== RUN CONFIGURATION ==============================
# Everything that controls the run lives in this cell.
# Paths can be overridden with the environment variables GLOBEM_DRIVE_ROOT (folder holding the PhysioNet zip),
# GLOBEM_DATA_ROOT (extracted data) and GLOBEM_OUTPUT_ROOT (outputs and checkpoints).
import os, sys, json, time, math, gc, re, glob, random, pickle, hashlib, warnings
import platform, datetime, shutil, importlib, importlib.util, subprocess, zipfile
from pathlib import Path
from collections import OrderedDict
from itertools import combinations
from contextlib import contextmanager

SMOKE_TEST = os.environ.get('GLOBEM_SMOKE_TEST', '0') == '1'   # tiny budgets: pipeline check only

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
else:
    DRIVE_ROOT = Path(os.environ.get('GLOBEM_DRIVE_ROOT', str(Path.cwd())))

ZIP_NAME = ('globem-dataset-multi-year-datasets-for-longitudinal-human-behavior-'
            'modeling-generalization-1.1.zip')
ZIP_PATH = DRIVE_ROOT / ZIP_NAME
EXTRACT_DIR = Path(os.environ.get('GLOBEM_DATA_ROOT', str(DRIVE_ROOT / 'GLOBEM_extracted')))

DATASET_YEARS = {'INS-W_1': 2018, 'INS-W_2': 2019, 'INS-W_3': 2020, 'INS-W_4': 2021}
LABEL_COLUMN = 'dep'            # weekly binary label used by the GLOBEM benchmark
FEATURE_FILE = 'rapids.csv'
SEQUENCE_DAYS = 28              # window ends on (and includes) the label date: detection, not forecasting
MIN_VALID_DAYS = 7
MAX_MISSING_FRAC = 0.95
VARIANCE_THRESHOLD = 0.01
CORR_THRESHOLD = 0.9
INNER_VAL_SIZE = 0.2            # participant-disjoint inner validation (tuning, early stopping, threshold)
PD_TEST_SIZE = 0.2              # participant-disjoint reference split
SPLIT_SEED = 42
BDI_THRESHOLD = 13              # end-of-term BDI-II > 13 (consistent across all four years)

EXCLUDE_POPULATION_NORM_FEATURES = False   # main analysis keeps GLOBEM's released `_norm` features
RUN_NORM_ABLATION = True                   # supplementary: tabular models with `_norm` removed

SEEDS = list(range(10))                    # >= 10 seeds for every model
TUNING_SEED = 1000                         # inner split used by Optuna (distinct from evaluation seeds)
N_BOOT = 2000                              # two-level bootstrap replicates
EQUIV_MARGIN = 0.02                        # AUC equivalence margin for "no meaningful difference"
ALPHA = 0.05

MODELS_ALL = ['Dummy', 'LogReg', 'XGBoost', 'MLP-ERM', 'MLP-GroupDRO', 'MLP-IRM', 'GRU', 'Transformer']
HPO_TRIALS = {'LogReg': 25, 'XGBoost': 40, 'MLP-ERM': 25, 'MLP-GroupDRO': 25, 'MLP-IRM': 25,
              'GRU': 20, 'Transformer': 20}
HPO_TIMEOUT_S = 2400                       # per (model, split) study; safety net only
MLP_MAX_EPOCHS, MLP_PATIENCE = 80, 15
SEQ_MAX_EPOCHS, SEQ_PATIENCE = 60, 10

RUN_SENSITIVITY_EX2018 = True              # full rerun without 2018 (PANAS-derived labels)
SENS_RETUNE = True                         # re-run Optuna on the 2018-free training sets
RUN_BDI_ENDTERM = True                     # consistent-outcome analysis (end-of-term BDI-II)
BDI_MODELS = ['Dummy', 'LogReg', 'XGBoost', 'MLP-ERM']
RUN_DOMAIN_SHIFT = True
DOMAIN_CV_FOLDS = 5
DOMAIN_DEP_SEEDS = 5                       # seeds for depression AUC on feature-subset variants
RUN_PERMUTATION_TEST = True
N_PERMUTATIONS = 200                       # label permutations per split (p-value resolution 1/201)
RUN_EXPLANATION_STABILITY = True
SPLIT_HALF_REPEATS = 10
YEAR_PARTITION_PERMUTATIONS = 200
TOP_K = 20
TOP_UNION = 100
PREVALENCES = [0.05, 0.10, 0.15, 0.20, 0.25]
YEAR_SHAP_ROWS = 3000                      # held-out windows explained (out-of-fold) for the year-classifier SHAP

if SMOKE_TEST:
    SEEDS = [0, 1]
    N_BOOT = 200
    HPO_TRIALS = {k: 2 for k in HPO_TRIALS}
    HPO_TIMEOUT_S = 120
    MLP_MAX_EPOCHS, MLP_PATIENCE = 4, 2
    SEQ_MAX_EPOCHS, SEQ_PATIENCE = 3, 2
    DOMAIN_CV_FOLDS = 3
    DOMAIN_DEP_SEEDS = 2
    N_PERMUTATIONS = 3
    SPLIT_HALF_REPEATS = 3
    YEAR_PARTITION_PERMUTATIONS = 20
    TOP_K, TOP_UNION = 5, 15
    YEAR_SHAP_ROWS = 300

RESUME = True                              # reuse checkpoints whose fingerprint still matches (see Setup)
PIPELINE_VERSION = 'jmir-r2'               # bump to force recomputation of every cached stage
RUN_NAME = ('SMOKE_' if SMOKE_TEST else '') + ('normEXCL' if EXCLUDE_POPULATION_NORM_FEATURES else 'normINCL')
OUTPUT_ROOT = Path(os.environ.get('GLOBEM_OUTPUT_ROOT', str(DRIVE_ROOT / 'GLOBEM_JMIR_outputs'))) / RUN_NAME

# Every setting, recorded in the run manifest. Cache invalidation does not rely on this dictionary: each cached stage
# is keyed by a fingerprint of exactly the code, settings and upstream results it depends on (Setup cell).
CONFIG = {k: v for k, v in globals().items() if k.isupper() and not k.startswith('_')
          and isinstance(v, (bool, int, float, str, list, tuple, dict))}
CONFIG_HASH = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:10]
CKPT_DIR = OUTPUT_ROOT / 'checkpoints'
MAIN_FIG_DIR = OUTPUT_ROOT / 'main_figures'
MAIN_TAB_DIR = OUTPUT_ROOT / 'main_tables'
SUPP_FIG_DIR = OUTPUT_ROOT / 'supplementary' / 'figures'
SUPP_TAB_DIR = OUTPUT_ROOT / 'supplementary' / 'tables'
for _d in (CKPT_DIR, MAIN_FIG_DIR, MAIN_TAB_DIR, SUPP_FIG_DIR, SUPP_TAB_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print('Smoke test :', SMOKE_TEST)
print('Data root  :', EXTRACT_DIR)
print('Outputs    :', OUTPUT_ROOT)
print('Checkpoints:', CKPT_DIR, '(each stage keyed by a fingerprint of its code, settings and inputs)')

In [ ]:
# ============================== SETUP ==============================
def _ensure(module, pip_name=None):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or module])

for _m in ['optuna', 'xgboost', 'tabulate', 'pyarrow']:
    _ensure(_m)

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components
from scipy.stats import spearmanr
from scipy.optimize import minimize_scalar
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score, f1_score, average_precision_score,
                             confusion_matrix, precision_score, brier_score_loss)
import xgboost as xgb
import torch
import torch.nn as nn
import optuna
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
xgb.set_config(verbosity=0)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

random.seed(SPLIT_SEED)
np.random.seed(SPLIT_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = False   # keep float32 precision for correlation thresholds
    torch.backends.cudnn.allow_tf32 = False
_XGB_MAJOR = int(xgb.__version__.split('.')[0])

def xgb_device_kwargs():
    if DEVICE.type == 'cuda':
        return {'device': 'cuda', 'tree_method': 'hist'} if _XGB_MAJOR >= 2 else {'tree_method': 'gpu_hist'}
    return {'tree_method': 'hist'}

print(f'torch {torch.__version__} | device {DEVICE} | xgboost {xgb.__version__} | optuna {optuna.__version__} '
      f'| pandas {pd.__version__} | numpy {np.__version__}')

# --- Checkpointing ----------------------------------------------------------
# Each cached object is stored as <stage>__<fingerprint>.pkl. The fingerprint hashes the source code of the functions
# that produce it, the settings it uses and the fingerprints of its inputs (data, splits, tuned hyperparameters,
# trained models). Editing code or settings therefore invalidates exactly the affected stages and everything
# downstream of them (editing only comments also invalidates, which is conservative), and an interrupted run resumes
# from the last valid stage. Library versions are stored with every checkpoint; a mismatch is reported.
import inspect, types
import sklearn
LIB_VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                'sklearn': sklearn.__version__, 'xgboost': xgb.__version__, 'torch': torch.__version__,
                'optuna': optuna.__version__}
CK_STATS = {'loaded': 0, 'computed': 0}
CK_VERSION_MISMATCH, FP_FALLBACKS = [], []

def _code_bytes(co):
    out = [co.co_code, repr(co.co_names).encode(), repr(co.co_varnames).encode()]
    for c in co.co_consts:
        out.append(_code_bytes(c) if isinstance(c, types.CodeType) else repr(c).encode())
    return b''.join(out)

def _canon(o):
    if isinstance(o, str):
        return repr(o)
    if isinstance(o, dict):
        return '{' + ','.join(f'{_canon(k)}:{_canon(v)}' for k, v in sorted(o.items(), key=lambda kv: repr(kv[0]))) + '}'
    if isinstance(o, (list, tuple)):
        return '[' + ','.join(_canon(v) for v in o) + ']'
    if isinstance(o, (set, frozenset)):
        return '{' + ','.join(sorted(_canon(v) for v in o)) + '}'
    if isinstance(o, np.ndarray):
        return f'nd{o.dtype}{o.shape}' + hashlib.md5(np.ascontiguousarray(o).tobytes()).hexdigest()
    if isinstance(o, (np.integer, np.floating, np.bool_)):
        return repr(o.item())
    if isinstance(o, (type, types.FunctionType, types.MethodType)):
        return hashlib.md5(_src_bytes(o)).hexdigest()
    return repr(o)

def _src_bytes(obj):
    # Source text of a function or class (methods included); bytecode fallback; canonical repr for data.
    if isinstance(obj, type):
        parts = [obj.__name__.encode()] + [_src_bytes(b) for b in obj.__bases__ if b.__module__ == '__main__']
        for k, v in sorted(vars(obj).items()):
            f = getattr(v, '__func__', v)
            if isinstance(f, types.FunctionType):
                parts.append(k.encode() + _src_bytes(f))
        return b'|'.join(parts)
    if isinstance(obj, (types.FunctionType, types.MethodType)):
        try:
            return inspect.getsource(obj).encode()
        except (OSError, TypeError):
            FP_FALLBACKS.append(getattr(obj, '__qualname__', repr(obj)))
            return _code_bytes(obj.__code__)
    return _canon(obj).encode()

def fp(*parts):
    h = hashlib.md5(PIPELINE_VERSION.encode())
    for p in parts:
        h.update(b'\x1f')
        h.update(_src_bytes(p))
    return h.hexdigest()[:12]

def _ck(name):
    return CKPT_DIR / (re.sub(r'[^A-Za-z0-9_.=+-]+', '_', name) + '.pkl')

_CK_MEM = {}
CK_LOAD_ERRORS = []

def ck_exists(name):
    # True only if the checkpoint exists AND can be read. An unreadable file (for example one written by an
    # incompatible library version) is reported and that stage is recomputed instead of stopping the run.
    if not (RESUME and _ck(name).exists()):
        return False
    if name in _CK_MEM:
        return True
    try:
        with open(_ck(name), 'rb') as f:
            d = pickle.load(f)
        if not (isinstance(d, dict) and 'obj' in d):
            raise ValueError('unexpected checkpoint format')
        _CK_MEM[name] = d
        return True
    except Exception as e:
        CK_LOAD_ERRORS.append(name)
        print(f'WARNING: checkpoint {name} could not be read ({type(e).__name__}: {str(e)[:120]}); recomputing this stage.')
        return False

def ck_load(name):
    if name not in _CK_MEM and not ck_exists(name):
        raise FileNotFoundError(f'checkpoint {name} is missing or unreadable')
    d = _CK_MEM.pop(name)
    if d.get('versions') != LIB_VERSIONS:
        CK_VERSION_MISMATCH.append(name)
        if len(CK_VERSION_MISMATCH) == 1:
            print(f'WARNING: checkpoint {name} was written with {d.get("versions")}; current {LIB_VERSIONS}.')
    CK_STATS['loaded'] += 1
    return d['obj']

def ck_save(name, obj):
    p = _ck(name)
    tmp = p.with_name(p.name + '.tmp')
    with open(tmp, 'wb') as f:
        pickle.dump({'obj': obj, 'versions': LIB_VERSIONS, 'saved': datetime.datetime.now().isoformat(timespec='seconds')},
                    f, protocol=pickle.HIGHEST_PROTOCOL)
    tmp.replace(p)
    CK_STATS['computed'] += 1

def cached(name, fn):
    if ck_exists(name):
        return ck_load(name)
    obj = fn()
    ck_save(name, obj)
    return obj

STAGE_TIMES = []

@contextmanager
def stage(name):
    t0 = time.time()
    print(f'\n=== {name} ===')
    yield
    dt = time.time() - t0
    STAGE_TIMES.append({'stage': name, 'seconds': round(dt, 1)})
    print(f'=== {name}: {dt/60:.1f} min ===')

def stable_seed(*parts):
    return int(hashlib.md5('|'.join(map(str, parts)).encode()).hexdigest()[:8], 16)

# --- Artefact register and savers -------------------------------------------
ARTEFACTS = []
matplotlib.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans'],
    'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9, 'legend.fontsize': 8,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'axes.grid': True, 'grid.alpha': 0.25,
    'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 110, 'savefig.bbox': 'tight'})
PALETTE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9', '#F0E442', '#000000']
MODEL_COLORS = {'Dummy': '#9E9E9E', 'LogReg': '#56B4E9', 'XGBoost': '#0072B2', 'MLP-ERM': '#009E73',
                'MLP-GroupDRO': '#CC79A7', 'MLP-IRM': '#E69F00', 'GRU': '#D55E00', 'Transformer': '#000000'}

def save_fig(fig, name, caption, main=False):
    d = MAIN_FIG_DIR if main else SUPP_FIG_DIR
    paths = []
    for ext in ('png', 'pdf'):
        p = d / f'{name}.{ext}'
        fig.savefig(p, dpi=300)
        paths.append(p)
    ARTEFACTS.append({'kind': 'figure', 'main': main, 'name': name, 'caption': caption, 'file': str(paths[0])})
    print(f'  [figure] {name}')

def save_table(df, name, caption, main=False, index=False):
    if df is None or len(df) == 0:
        print(f'  [table ] {name}: skipped (empty)')
        return
    d = MAIN_TAB_DIR if main else SUPP_TAB_DIR
    df.to_csv(d / f'{name}.csv', index=index)
    try:
        md = df.to_markdown(index=index, floatfmt='.4f')
    except Exception:
        md = df.to_string(index=index)
    (d / f'{name}.md').write_text(f'**{caption}**\n\n{md}\n', encoding='utf-8')
    ARTEFACTS.append({'kind': 'table', 'main': main, 'name': name, 'caption': caption, 'file': str(d / f'{name}.csv')})
    print(f'  [table ] {name} ({df.shape[0]}x{df.shape[1]})')

KEY_NUMBERS = {}

## 1. Load the four GLOBEM cohorts
Weekly labels (`dep_weekly.csv`), end-of-term BDI-II (`dep_endterm.csv`), daily RAPIDS features and
device platform. The 2018 weekly file contains PANAS items instead of PHQ-4; this is recorded per year.

In [ ]:
def find_dataset_folders(root, expected):
    if not root.exists():
        if ZIP_PATH.exists():
            root.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(ZIP_PATH) as z:
                z.extractall(root)
        else:
            raise FileNotFoundError(f'Neither {root} nor {ZIP_PATH} exists.')
    found = {}
    cands = [p for p in root.rglob('*') if p.is_dir() and p.name in expected]
    if root.name in expected:
        cands.append(root)
    for p in cands:
        if (p / 'SurveyData').exists() and (p / 'FeatureData').exists():
            found[p.name] = p
    missing = sorted(set(expected) - set(found))
    if missing:
        raise FileNotFoundError(f'Missing dataset folders: {missing}')
    return {k: found[k] for k in sorted(found)}

_INDEX_LIKE_RE = re.compile(r'^(unnamed(?::\s*\d+)?|index|level_\d+)$', re.I)

def _drop_index_like(df):
    return df.drop(columns=[c for c in df.columns if _INDEX_LIKE_RE.match(str(c).strip())])

def _find_col(df, names, contains=None):
    low = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in low:
            return low[n.lower()]
    if contains:
        for c in df.columns:
            if contains in c.lower():
                return c
    raise ValueError(f'No column among {names} in {list(df.columns)[:15]}')

def _norm_pid(s):
    return s.map(lambda x: np.nan if pd.isna(x) else str(x).strip())

def _as_binary(s):
    # Robust boolean parsing (bool, 0/1, 'True'/'False' strings); avoids astype(bool) treating 'False' as True.
    return s.map(lambda v: str(v).strip().lower() in ('true', '1', '1.0', 'yes')).astype(int)

def read_features(ds, path):
    df = _drop_index_like(pd.read_csv(path / 'FeatureData' / FEATURE_FILE))
    df = df.rename(columns={_find_col(df, ['pid', 'participant_id', 'user_id']): 'pid',
                            _find_col(df, ['date'], 'date'): 'date'})
    df['pid'] = _norm_pid(df['pid'])
    df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.normalize()
    df['dataset'] = ds
    df['year'] = DATASET_YEARS[ds]
    return df.dropna(subset=['pid', 'date'])

def read_weekly_labels(ds, path):
    df = _drop_index_like(pd.read_csv(path / 'SurveyData' / 'dep_weekly.csv'))
    cols = set(df.columns)
    source = ('PHQ-4 (direct)' if 'phq4' in cols else
              'PANAS-derived (decision tree)' if {'feel_depressed', 'feel_anxious'} & cols else 'unknown')
    df = df.rename(columns={_find_col(df, ['pid']): 'pid', _find_col(df, ['date'], 'date'): 'date',
                            LABEL_COLUMN: 'label'})
    df['pid'] = _norm_pid(df['pid'])
    df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.normalize()
    df = df.dropna(subset=['pid', 'date', 'label'])
    df['label'] = _as_binary(df['label'])
    df['dataset'] = ds
    df['year'] = DATASET_YEARS[ds]
    info = {'Year': DATASET_YEARS[ds], 'Dataset': ds, 'Weekly label source': source,
            'Survey columns': ', '.join(sorted(c for c in cols if c not in ('pid', 'date'))),
            'Weekly rows': len(df), 'Weekly positive rate': round(float(df['label'].mean()), 4)}
    return df[['pid', 'date', 'dataset', 'year', 'label']], info

def read_endterm_labels(ds, path):
    # End-of-term BDI-II > BDI_THRESHOLD. Strict: never falls back to any other label definition.
    skip = ' Set RUN_BDI_ENDTERM = False to run without the consistent-outcome analysis.'
    p = path / 'SurveyData' / 'dep_endterm.csv'
    if not p.exists():
        raise FileNotFoundError(f'{ds}: {p} not found; the BDI-II analysis needs end-term scores in every year.' + skip)
    df = _drop_index_like(pd.read_csv(p))
    bdi_col = next((c for c in df.columns if str(c).lower() in ('bdi2', 'bdi_ii', 'bdi-ii')), None)
    if bdi_col is None:
        raise ValueError(f'{ds}: no BDI-II score column in {p.name} (columns: {list(df.columns)}).' + skip)
    df = df.rename(columns={_find_col(df, ['pid']): 'pid', _find_col(df, ['date'], 'date'): 'date', bdi_col: 'BDI2'})
    n_rows = len(df)
    df['pid'] = _norm_pid(df['pid'])
    df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.normalize()
    df['BDI2'] = pd.to_numeric(df['BDI2'], errors='coerce')
    df = df.dropna(subset=['pid', 'date', 'BDI2'])
    if len(df) == 0:
        raise ValueError(f'{ds}: no rows with a numeric BDI-II score and a date.' + skip)
    bad = int(((df['BDI2'] < 0) | (df['BDI2'] > 63)).sum())
    if bad:
        raise ValueError(f'{ds}: {bad} BDI-II scores outside the instrument range 0-63.' + skip)
    n_dup = int(df.duplicated('pid').sum())
    df = df.sort_values('date', kind='mergesort').drop_duplicates('pid', keep='last')
    df['label'] = (df['BDI2'] > BDI_THRESHOLD).astype(int)
    agree = float((_as_binary(df['dep']) == df['label']).mean()) if 'dep' in df.columns else np.nan
    print(f'  {ds}: end-term label = BDI-II > {BDI_THRESHOLD} (n={len(df)}); agreement with released `dep`: {agree:.3f}')
    df['dataset'] = ds
    df['year'] = DATASET_YEARS[ds]
    info = {'Year': DATASET_YEARS[ds], 'Dataset': ds, 'End-term rows in file': n_rows, 'Participants with BDI-II': len(df),
            'Duplicate rows dropped': n_dup, f'BDI-II > {BDI_THRESHOLD} rate': round(float(df['label'].mean()), 4),
            'Agreement with released dep': round(agree, 4) if np.isfinite(agree) else np.nan}
    return df[['pid', 'date', 'dataset', 'year', 'label', 'BDI2']], info

def read_platform(ds, path):
    p = path / 'ParticipantsInfoData' / 'platform.csv'
    if not p.exists():
        return pd.DataFrame(columns=['pid', 'dataset', 'platform'])
    df = _drop_index_like(pd.read_csv(p))
    df = df.rename(columns={_find_col(df, ['pid']): 'pid'})
    pcol = [c for c in df.columns if c != 'pid'][0]
    df['pid'] = _norm_pid(df['pid'])
    df['platform'] = df[pcol].astype(str).str.lower()
    df['dataset'] = ds
    return df[['pid', 'dataset', 'platform']]

with stage('Load raw data'):
    dataset_paths = find_dataset_folders(EXTRACT_DIR, DATASET_YEARS)
    _f, _l, _e, _p, _info, _binfo = [], [], [], [], [], []
    for ds, path in dataset_paths.items():
        print('Loading', ds)
        _f.append(read_features(ds, path))
        lab, info = read_weekly_labels(ds, path)
        _l.append(lab); _info.append(info)
        if RUN_BDI_ENDTERM:
            _et_df, _et_info = read_endterm_labels(ds, path)
            _e.append(_et_df); _binfo.append(_et_info)
        _p.append(read_platform(ds, path))
    features = pd.concat(_f, ignore_index=True); del _f
    labels = pd.concat(_l, ignore_index=True)
    endterm = (pd.concat(_e, ignore_index=True) if _e else
               pd.DataFrame(columns=['pid', 'date', 'dataset', 'year', 'label', 'BDI2']))
    bdi_source_table = pd.DataFrame(_binfo)
    platform_raw = pd.concat(_p, ignore_index=True)
    label_source_table = pd.DataFrame(_info)
    gc.collect()
print('features', features.shape, '| weekly labels', labels.shape, '| end-term', endterm.shape)
display(label_source_table)

# Device platform per participant (iOS / Android / both).
def _platform_of(s):
    v = set(s)
    has_i = any('ios' in x or 'iphone' in x for x in v)
    has_a = any('android' in x for x in v)
    return 'both' if (has_i and has_a) else 'ios' if has_i else 'android' if has_a else 'unknown'
platform_map = (platform_raw.groupby(['dataset', 'pid'])['platform'].agg(_platform_of).to_dict()
                if len(platform_raw) else {})

# Identifier audit: count participant IDs shared between years. With no shared IDs, repeat participation cannot
# be verified from the released data.
_pidsets = {y: set(labels.loc[labels['year'] == y, 'pid']) for y in sorted(labels['year'].unique())}
pid_linkage_table = pd.DataFrame([{'Year A': a, 'Year B': b, 'Overlapping IDs': len(_pidsets[a] & _pidsets[b])}
                                  for a, b in combinations(sorted(_pidsets), 2)])
PID_LINKED = pid_linkage_table['Overlapping IDs'].sum() > 0
print('PID overlap across years:', int(pid_linkage_table['Overlapping IDs'].sum()),
      '-> cross-year splits are year-held-out with UNVERIFIABLE user overlap' if not PID_LINKED else '')

### Label audit
Checks that the released `dep` label equals PHQ-4 > 2 on weekly rows with a PHQ-4 score, and BDI-II > 13 on rows
that carry a BDI-II score instead. Rows with neither score are counted but cannot be checked.

In [ ]:
# Agreement between the released binary label `dep` and the questionnaire thresholds it is documented to follow.
for ds, path in dataset_paths.items():
    w = pd.read_csv(path / 'SurveyData' / 'dep_weekly.csv')
    if 'phq4' in w.columns:
        w = w.dropna(subset=['phq4', 'dep'])
        agree = ((w['phq4'] > 2).astype(int) == _as_binary(w['dep'])).mean()
        print(f'{ds}: {len(w)} rows with a PHQ-4 score | agreement of dep with PHQ-4 > 2: {agree:.4f}')
    else:
        print(f'{ds}: no PHQ-4 column in dep_weekly.csv')

# Labelled rows without a PHQ-4 score: how many carry a BDI-II score, and does `dep` follow BDI-II > BDI_THRESHOLD?
for ds, path in dataset_paths.items():
    w = pd.read_csv(path / 'SurveyData' / 'dep_weekly.csv').dropna(subset=['dep'])
    no_phq = w['phq4'].isna() if 'phq4' in w.columns else pd.Series(True, index=w.index)
    sub = w[no_phq]
    has_bdi = sub['BDI2'].notna() if 'BDI2' in sub.columns else pd.Series(False, index=sub.index)
    agree = ((sub.loc[has_bdi, 'BDI2'] > BDI_THRESHOLD).astype(int) == _as_binary(sub.loc[has_bdi, 'dep'])).mean()
    print(f'{ds}: {len(w)} labelled rows | without PHQ-4: {int(no_phq.sum())} | of which with BDI-II: {int(has_bdi.sum())} '
          f'| agreement of dep with BDI-II > {BDI_THRESHOLD}: {agree:.4f}')

## 2. Feature cleaning and daily missingness

In [ ]:
MODALITY_PATTERNS = [('Location', ('f_loc',)), ('Screen', ('f_screen', 'f_scr')), ('Calls', ('f_call',)),
                     ('Bluetooth', ('f_blue', 'f_bt')), ('Steps', ('f_steps', 'f_step')),
                     ('Sleep', ('f_slp', 'f_sleep')), ('Wi-Fi', ('f_wifi',))]
MODALITIES = [m for m, _ in MODALITY_PATTERNS]

def modality_of(col):
    c = str(col).lower()
    if c.startswith('hist_avail:'):
        c = c.split(':', 1)[1]
    for label, prefixes in MODALITY_PATTERNS:
        if any(c.startswith(p) for p in prefixes):
            return label
    return 'History' if str(col).startswith('hist_avail:') else 'Other'

with stage('Clean features'):
    ID_COLS = ['pid', 'date', 'dataset', 'year']
    features = _drop_index_like(features)
    numeric_feature_cols = [c for c in features.columns if c not in ID_COLS
                            and pd.api.types.is_numeric_dtype(features[c])]
    # Daily missingness by modality x year (supplementary + domain-shift section).
    _rows = {}
    for mod in MODALITIES:
        cols_m = [c for c in numeric_feature_cols if modality_of(c) == mod]
        if cols_m:
            _rows[mod] = {int(y): float(features.loc[features['year'] == y, cols_m].isna().to_numpy().mean())
                          for y in sorted(features['year'].unique())}
    daily_missingness = pd.DataFrame(_rows).T
    feat_id = features[ID_COLS].copy()
    FEAT_VALUES = features[numeric_feature_cols].to_numpy(dtype=np.float32)
    del features; gc.collect()
    FEAT_VALUES[~np.isfinite(FEAT_VALUES)] = np.nan
    # Sort by participant/date and drop duplicate days (keep last).
    order = np.lexsort((feat_id['date'].values, feat_id['pid'].values, feat_id['dataset'].values))
    feat_id = feat_id.iloc[order].reset_index(drop=True)
    FEAT_VALUES = FEAT_VALUES[order]
    dup = feat_id.duplicated(['dataset', 'pid', 'date'], keep='last').to_numpy()
    if dup.any():
        feat_id = feat_id.loc[~dup].reset_index(drop=True)
        FEAT_VALUES = FEAT_VALUES[~dup]
    # Daily-missing counts per participant-year and channel. The channel filter (drop channels with > 95% missing
    # daily values) is applied PER SPLIT on the outer-training participants only (Section 6), so forward-chaining never
    # uses the test year's missingness. Here only the candidate superset is kept: channels passing the rule within at
    # least one year (a channel failing in every year cannot pass for any training set built from whole years).
    _keys = (feat_id['dataset'].astype(str) + '|' + feat_id['pid'].astype(str)).to_numpy()
    _starts = np.r_[0, np.flatnonzero(_keys[1:] != _keys[:-1]) + 1]
    PP_KEYS = _keys[_starts]
    PP_INDEX = {k: i for i, k in enumerate(PP_KEYS)}
    PP_ROWS = np.diff(np.r_[_starts, len(_keys)]).astype(np.int64)
    PP_YEAR = feat_id['year'].to_numpy()[_starts].astype(int)
    PP_NAN = np.zeros((len(PP_KEYS), FEAT_VALUES.shape[1]), dtype=np.int32)
    for _j in range(0, FEAT_VALUES.shape[1], 512):
        PP_NAN[:, _j:_j + 512] = np.add.reduceat(np.isnan(FEAT_VALUES[:, _j:_j + 512]).astype(np.int32), _starts, axis=0)
    _year_miss = np.vstack([PP_NAN[PP_YEAR == y].sum(0) / max(int(PP_ROWS[PP_YEAR == y].sum()), 1)
                            for y in sorted(np.unique(PP_YEAR))])
    _pooled_miss = PP_NAN.sum(0) / max(int(PP_ROWS.sum()), 1)
    _cand = (_year_miss <= MAX_MISSING_FRAC).any(axis=0)
    keep_feature_cols = [c for c, k in zip(numeric_feature_cols, _cand) if k]
    FEAT_VALUES = np.ascontiguousarray(FEAT_VALUES[:, _cand])
    PP_NAN = np.ascontiguousarray(PP_NAN[:, _cand])
    N_POOLED_RULE = int((_pooled_miss <= MAX_MISSING_FRAC).sum())
    del _keys; gc.collect()
print(f'Numeric columns {len(numeric_feature_cols)} | candidate channels (<= {MAX_MISSING_FRAC:.0%} missing in at least one '
      f'year) {len(keep_feature_cols)} | a pooled all-years rule would keep {N_POOLED_RULE}')
display((100 * daily_missingness).round(1))

## 3. 28-day windows (held in memory) and the aggregated feature table
Each window ends on and includes the label date (same-week detection). Windows with fewer than seven
observed days are dropped. Sequences are built once and held in memory, which keeps tuning and ten-seed
refitting affordable.

In [ ]:
def build_windows(feat_id, values, label_df, seq_days=SEQUENCE_DAYS, min_valid=MIN_VALID_DAYS):
    keys = (feat_id['dataset'].astype(str) + '|' + feat_id['pid'].astype(str)).to_numpy()
    starts = np.r_[0, np.flatnonzero(keys[1:] != keys[:-1]) + 1]
    bounds = {keys[s]: (s, e) for s, e in zip(starts, np.r_[starts[1:], len(keys)])}
    lab = label_df.copy()
    lab['_key'] = lab['dataset'].astype(str) + '|' + lab['pid'].astype(str)
    lab = lab.sort_values(['dataset', 'pid', 'date'], kind='mergesort')
    C = values.shape[1]
    out = np.empty((len(lab), seq_days, C), dtype=np.float32)
    rows, n = [], 0
    dates_all = feat_id['date'].to_numpy()
    for key, g in lab.groupby('_key', sort=False):
        if key not in bounds:
            continue
        s, e = bounds[key]
        d = dates_all[s:e].astype('datetime64[D]')
        ld = g['date'].to_numpy().astype('datetime64[D]')
        d0 = min(d.min(), ld.min() - np.timedelta64(seq_days - 1, 'D'))
        span = int((max(d.max(), ld.max()) - d0) / np.timedelta64(1, 'D')) + 1
        dense = np.full((span, C), np.nan, dtype=np.float32)
        dense[(d - d0).astype(int)] = values[s:e]
        for r, end_day in zip(g.itertuples(index=False), (ld - d0).astype(int)):
            w = dense[end_day - seq_days + 1:end_day + 1]
            valid = int((~np.isnan(w)).any(axis=1).sum())
            if valid < min_valid:
                continue
            out[n] = w
            rows.append({'pid': r.pid, 'dataset': r.dataset, 'year': int(r.year), 'label_date': r.date,
                         'label': int(r.label), 'valid_days': valid,
                         'missing_fraction': float(np.isnan(w).mean())})
            n += 1
    return out[:n], pd.DataFrame(rows)

def aggregate_windows(seq, feature_cols, block=256):
    n, _, C = seq.shape
    agg = np.empty((n, 4 * C), dtype=np.float32)
    with warnings.catch_warnings(), np.errstate(all='ignore'):
        warnings.simplefilter('ignore', RuntimeWarning)
        for i in range(0, n, block):
            X = seq[i:i + block]
            b = X.shape[0]
            agg[i:i + b, :C] = np.nanmean(X, axis=1)
            agg[i:i + b, C:2 * C] = np.nanstd(X, axis=1)
            agg[i:i + b, 2 * C:3 * C] = X[:, -1, :]
            agg[i:i + b, 3 * C:] = np.isnan(X).mean(axis=1)
    agg[~np.isfinite(agg)] = np.nan
    cols = ([f'{c}_mean' for c in feature_cols] + [f'{c}_std' for c in feature_cols] +
            [f'{c}_last' for c in feature_cols] + [f'{c}_missing' for c in feature_cols])
    return agg, cols

def _build_weekly():
    seq, meta_ = build_windows(feat_id, FEAT_VALUES, labels)
    agg, cols = aggregate_windows(seq, keep_feature_cols)
    return seq, meta_, agg, cols

with stage('Build windows'):
    SEQ, meta, _agg, AGG_COLS = _build_weekly()
    ml_table = pd.concat([meta.reset_index(drop=True), pd.DataFrame(_agg, columns=AGG_COLS)], axis=1)
    del _agg; gc.collect()
print('windows:', SEQ.shape, '| ml_table:', ml_table.shape)
display(meta.groupby('year')['label'].agg(['count', 'mean']))

### Causal within-person deviation features
Deviation from the expanding mean of the participant's *prior* windows only, restricted to level-type
constructs, plus participant- and modality-level history-availability indicators.

In [ ]:
SELFDEV_EXCLUDE_TOKENS = ('_norm:', 'variance', 'std', 'entropy', 'ratio')
ALL_KNOWN_WINDOWS = {'allday', 'morning', 'afternoon', 'evening', 'night', 'weekday', 'weekend', '7dhist', '14dhist'}

def _window_token(col):
    base = col
    for sfx in ('_mean', '_std', '_last', '_missing'):
        if base.endswith(sfx):
            base = base[:-len(sfx)]
            break
    if ':' not in base:
        return None
    tail = base.rsplit(':', 1)[-1]
    return tail if tail in ALL_KNOWN_WINDOWS else None

def _is_selfdev_eligible(c):
    if not (c.endswith('_last') or c.endswith('_mean')):
        return False
    low = c.lower()
    if any(t in low for t in SELFDEV_EXCLUDE_TOKENS):
        return False
    return _window_token(c) not in {'7dhist', '14dhist'}

def causal_selfdev(df, feature_cols, min_history=1):
    sdf = df.sort_values(['dataset', 'pid', 'label_date'], kind='mergesort')
    grp = pd.factorize(sdf['dataset'].astype(str) + '|' + sdf['pid'].astype(str))[0]
    elig = [c for c in feature_cols if _is_selfdev_eligible(c)]
    X = sdf[elig].to_numpy(dtype=np.float64)
    obs = ~np.isnan(X)
    Xz = np.where(obs, X, 0.0)
    cs = pd.DataFrame(Xz).groupby(grp).cumsum().to_numpy()
    cc = pd.DataFrame(obs.astype(np.float64)).groupby(grp).cumsum().to_numpy()
    prior_sum, prior_cnt = cs - Xz, cc - obs
    with np.errstate(all='ignore'):
        hist = np.where(prior_cnt >= min_history, prior_sum / np.maximum(prior_cnt, 1), np.nan)
    new = {f'{c}_selfdev_causal': (X[:, j] - hist[:, j]).astype(np.float32) for j, c in enumerate(elig)}
    cnt = pd.Series(np.ones(len(sdf))).groupby(grp).cumsum().to_numpy() - 1
    new['hist_avail:prior_observation_count'] = cnt.astype(np.float32)
    new['hist_avail:has_prior_observation'] = (cnt > 0).astype(np.float32)
    mods = {}
    for j, c in enumerate(elig):
        mods.setdefault(c.split(':', 1)[0], []).append(j)
    for m, js in mods.items():
        now = obs[:, js].any(axis=1).astype(np.float64)
        prior = pd.Series(now).groupby(grp).cumsum().to_numpy() - now
        new[f'hist_avail:{m}'] = (prior >= min_history).astype(np.float32)
    out = pd.DataFrame(new, index=sdf.index).loc[df.index]
    return out, [k for k in new if k.endswith('_selfdev_causal')], [k for k in new if k.startswith('hist_avail:')]

with stage('Self-deviation features'):
    _sd, SELFDEV_COLS, AVAIL_COLS = causal_selfdev(ml_table, AGG_COLS)
    ml_table = pd.concat([ml_table, _sd], axis=1); del _sd; gc.collect()

FEATURES_ALL = AGG_COLS + SELFDEV_COLS + AVAIL_COLS
FEATURES_NONORM = [c for c in FEATURES_ALL if '_norm:' not in c.lower()]
MAIN_FEATURES = FEATURES_NONORM if EXCLUDE_POPULATION_NORM_FEATURES else FEATURES_ALL
NON_FEATURE_COLS = ['pid', 'dataset', 'year', 'label_date', 'label', 'valid_days', 'missing_fraction']
ml_table['platform'] = [platform_map.get((d, p), 'unknown') for d, p in zip(ml_table['dataset'], ml_table['pid'])]
print(f'Candidate features: all {len(FEATURES_ALL)} | without _norm {len(FEATURES_NONORM)} | '
      f'main analysis uses {len(MAIN_FEATURES)}')

# Sequence channels for the GRU/Transformer: raw daily channels underlying the main tabular features (the per-split
# training-only missingness rule is applied on top of this mask in Section 6).
_agg_set = set(AGG_COLS)
_raw_used = {re.sub(r'_(mean|std|last|missing)$', '', c) for c in MAIN_FEATURES if c in _agg_set}
SEQ_MAIN_MASK = np.array([c in _raw_used for c in keep_feature_cols])
print('Candidate sequence channels:', int(SEQ_MAIN_MASK.sum()))

# Cohort table (main text / supplement).
cohort_table = (meta.groupby('year').agg(Participants=('pid', 'nunique'), Windows=('label', 'size'),
                                          Positive_rate=('label', 'mean'), Observed_days=('valid_days', 'mean'))
                .reset_index().rename(columns={'year': 'Year'}))
cohort_table = cohort_table.merge(label_source_table[['Year', 'Weekly label source', 'Weekly rows']], on='Year')
_et = (endterm.groupby('year').agg(BDI_participants=('pid', 'nunique'), BDI_positive_rate=('label', 'mean')).reset_index()
       if len(endterm) else pd.DataFrame({'year': pd.Series(dtype=int), 'BDI_participants': pd.Series(dtype=int),
                                          'BDI_positive_rate': pd.Series(dtype=float)}))
cohort_table = cohort_table.merge(_et.rename(columns={'year': 'Year'}), on='Year', how='left')
display(cohort_table)

## 4. End-of-term BDI-II table (consistent outcome across all four years)
One window per participant-term, ending on the end-of-term survey date. Self-deviation features are not
defined here (no prior labelled windows), so only the aggregated window features are used.

In [ ]:
BDI_FEATURES = []
bdi_table = None
if RUN_BDI_ENDTERM:
    with stage('Build BDI-II end-term windows'):
        _seq_b, bdi_meta = build_windows(feat_id, FEAT_VALUES, endterm)
        _agg_b, _ = aggregate_windows(_seq_b, keep_feature_cols)
        bdi_table = pd.concat([bdi_meta.reset_index(drop=True), pd.DataFrame(_agg_b, columns=AGG_COLS)], axis=1)
        del _seq_b, _agg_b; gc.collect()
        BDI_FEATURES = [c for c in AGG_COLS if (not EXCLUDE_POPULATION_NORM_FEATURES) or '_norm:' not in c.lower()]
    print('BDI-II table:', bdi_table.shape)
    _bw = bdi_meta.groupby('year')['label'].agg(['size', 'mean'])
    _missing_years = [y for y in sorted(meta['year'].unique()) if y not in _bw.index]
    if _missing_years:
        raise ValueError(f'BDI-II analysis: no end-term window with >= {MIN_VALID_DAYS} observed days in {_missing_years}. '
                         'The consistent-outcome analysis needs every year; set RUN_BDI_ENDTERM = False to skip it.')
    for _y, _r in _bw.iterrows():
        if _r['mean'] in (0.0, 1.0):
            print(f'WARNING: BDI-II windows in {_y} contain a single class; AUC is undefined for that test year.')
    bdi_source_table['Windows retained'] = bdi_source_table['Year'].map(_bw['size'])
    bdi_source_table['Window positive rate'] = bdi_source_table['Year'].map(_bw['mean']).round(4)
    display(bdi_source_table)
del FEAT_VALUES; gc.collect()

FRAMES = {'ml': ml_table}
if bdi_table is not None:
    FRAMES['bdi'] = bdi_table
FEATURE_SETS = {'main': MAIN_FEATURES, 'nonorm': FEATURES_NONORM, 'bdi': BDI_FEATURES}
_FM_CACHE = {}

def feature_matrix(frame, fs):
    key = (frame, fs)
    if key not in _FM_CACHE:
        _FM_CACHE[key] = FRAMES[frame][FEATURE_SETS[fs]].to_numpy(dtype=np.float32)
    return _FM_CACHE[key]

## 5. Evaluation protocols (split registry)
* **Forward-chaining (primary):** train only on years before the test year.
* **Leave-one-year-out:** domain-transfer reference (later years may be used for training).
* **Participant-disjoint 80/20:** within-population reference.

Identical train/test index sets are trained once; the 2021 forward split is reported in both the
forward-chaining and LOYO families. Analyses: `main`, `ex2018` (2018 excluded everywhere),
`bdi` (end-of-term BDI-II), `normabl` (main splits, `_norm` removed), `era` and `domainvar` (Section 12).

In [ ]:
PROTO_FWD, PROTO_LOYO, PROTO_PD, PROTO_ERA = 'Forward-chaining', 'Leave-one-year-out', 'Participant-disjoint', 'Era transfer'
SPLITS, VIEWS, _SIG = {}, [], {}

def register_split(analysis, frame, protocol, view, test_year, name, tr, te, train_years):
    tr, te = np.sort(np.asarray(tr, dtype=np.int64)), np.sort(np.asarray(te, dtype=np.int64))
    key = (analysis, frame, tr.tobytes(), te.tobytes())
    if key in _SIG:
        name = _SIG[key]
    else:
        f = FRAMES[frame]
        overlap = set(f['pid'].iloc[tr]) & set(f['pid'].iloc[te])
        assert not overlap, f'participant leakage in {name}'
        SPLITS[name] = dict(analysis=analysis, frame=frame, train_idx=tr, test_idx=te,
                            train_years=sorted(int(y) for y in np.unique(f['year'].to_numpy()[tr])),
                            test_years=sorted(int(y) for y in np.unique(f['year'].to_numpy()[te])))
        _SIG[key] = name
    VIEWS.append(dict(analysis=analysis, protocol=protocol, view=view, test_year=test_year, split=name))
    return name

def build_protocol_splits(analysis, frame, years_allowed):
    f = FRAMES[frame]
    yrs = f['year'].to_numpy()
    ok = np.isin(yrs, years_allowed)
    for y in sorted(years_allowed):
        te = np.flatnonzero(ok & (yrs == y))
        tr = np.flatnonzero(ok & (yrs < y))
        if len(tr) and len(te):
            register_split(analysis, frame, PROTO_FWD, f'Forward \u2192 {y}', y, f'{analysis}|fwd_{y}', tr, te, None)
    for y in sorted(years_allowed):
        te = np.flatnonzero(ok & (yrs == y))
        tr = np.flatnonzero(ok & (yrs != y))
        if len(tr) and len(te):
            register_split(analysis, frame, PROTO_LOYO, f'Leave-out {y}', y, f'{analysis}|loyo_{y}', tr, te, None)
    idx = np.flatnonzero(ok)
    gss = GroupShuffleSplit(n_splits=1, test_size=PD_TEST_SIZE, random_state=SPLIT_SEED)
    a, b = next(gss.split(idx, f['label'].to_numpy()[idx], f['pid'].to_numpy()[idx]))
    register_split(analysis, frame, PROTO_PD, 'Participant-disjoint 80/20', None, f'{analysis}|pd', idx[a], idx[b], None)

YEARS = sorted(int(y) for y in meta['year'].unique())
YEARS_EX18 = [y for y in YEARS if y != 2018]
build_protocol_splits('main', 'ml', YEARS)
if RUN_SENSITIVITY_EX2018 and 2018 in YEARS:
    build_protocol_splits('ex2018', 'ml', YEARS_EX18)
if 'bdi' in FRAMES:
    build_protocol_splits('bdi', 'bdi', YEARS)

ANALYSES = OrderedDict()
ANALYSES['main'] = dict(feature_set='main', models=MODELS_ALL, tune=True, params_from='main', views_from='main', shap=True)
if RUN_SENSITIVITY_EX2018 and 2018 in YEARS:
    ANALYSES['ex2018'] = dict(feature_set='main', models=MODELS_ALL, tune=SENS_RETUNE,
                              params_from='ex2018' if SENS_RETUNE else 'main', views_from='ex2018', shap=True)
if 'bdi' in FRAMES:
    ANALYSES['bdi'] = dict(feature_set='bdi', models=BDI_MODELS, tune=True, params_from='bdi', views_from='bdi', shap=False)
if RUN_NORM_ABLATION and not EXCLUDE_POPULATION_NORM_FEATURES:
    ANALYSES['normabl'] = dict(feature_set='nonorm', models=['LogReg', 'XGBoost', 'MLP-ERM'], tune=False,
                               params_from='main', views_from='main', shap=False)

VIEW_ALIAS = {}

def views_of(analysis, protocol=None):
    src = ANALYSES[analysis]['views_from'] if analysis in ANALYSES else VIEW_ALIAS.get(analysis, analysis)
    v = [r for r in VIEWS if r['analysis'] == src]
    return [r for r in v if protocol is None or r['protocol'] == protocol]

def splits_of(analysis):
    src = ANALYSES[analysis]['views_from']
    out = []
    for r in views_of(src):
        if r['split'] not in out:
            out.append(r['split'])
    return out

split_composition = pd.DataFrame([{
    'Analysis': r['analysis'], 'Protocol': r['protocol'], 'View': r['view'], 'Computed split': r['split'],
    'Train years': ', '.join(map(str, SPLITS[r['split']]['train_years'])),
    'Test years': ', '.join(map(str, SPLITS[r['split']]['test_years'])),
    'Train windows': len(SPLITS[r['split']]['train_idx']), 'Test windows': len(SPLITS[r['split']]['test_idx']),
    'Train participants': FRAMES[SPLITS[r['split']]['frame']]['pid'].iloc[SPLITS[r['split']]['train_idx']].nunique(),
    'Test participants': FRAMES[SPLITS[r['split']]['frame']]['pid'].iloc[SPLITS[r['split']]['test_idx']].nunique(),
    'Test positive rate': round(float(FRAMES[SPLITS[r['split']]['frame']]['label'].iloc[SPLITS[r['split']]['test_idx']].mean()), 3),
} for r in VIEWS])
display(split_composition)

## 6. Core utilities: metrics, thresholds, train-only preprocessing, GPU sequence store
Raw channels with > 95% missing daily values are dropped per split, using only the outer-training participants'
daily data, so forward-chaining never uses the test year's missingness.

In [ ]:
def safe_auc(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    ok = np.isfinite(p)
    if ok.sum() == 0 or len(np.unique(y[ok])) < 2:
        return np.nan
    return float(roc_auc_score(y[ok], p[ok]))

def metric_dict(y_true, pred, proba):
    y_true, pred, proba = np.asarray(y_true).astype(int), np.asarray(pred).astype(int), np.asarray(proba, float)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    two = len(np.unique(y_true)) == 2
    return {'roc_auc': roc_auc_score(y_true, proba) if two else np.nan,
            'pr_auc': average_precision_score(y_true, proba) if two else np.nan,
            'balanced_accuracy': balanced_accuracy_score(y_true, pred), 'f1': f1_score(y_true, pred, zero_division=0),
            'sensitivity': tp / max(tp + fn, 1), 'specificity': tn / max(tn + fp, 1),
            'precision': precision_score(y_true, pred, zero_division=0),
            'brier': brier_score_loss(y_true, np.clip(proba, 0, 1)),
            'pred_positive_rate': float(pred.mean()), 'true_positive_rate': float(y_true.mean()),
            'prob_std': float(np.std(proba)), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}

def choose_validation_threshold(y_true, proba):
    """Balanced-accuracy-optimal threshold on validation data; ties -> closest to 0.5 (vectorised)."""
    y, p = np.asarray(y_true).astype(int), np.asarray(proba, float)
    ok = np.isfinite(p)
    y, p = y[ok], p[ok]
    if len(y) == 0 or len(np.unique(y)) < 2:
        return 0.5
    cand = np.unique(np.r_[0.0, p, 1.0])
    ps, ns = np.sort(p[y == 1]), np.sort(p[y == 0])
    tp = len(ps) - np.searchsorted(ps, cand, side='left')
    tn = np.searchsorted(ns, cand, side='left')
    ba = 0.5 * (tp / len(ps) + tn / len(ns))
    near = np.flatnonzero(ba >= ba.max() - 1e-12)
    return float(cand[near[np.argmin(np.abs(cand[near] - 0.5))]])

def make_group_inner_split(indices, frame, seed, val_size=INNER_VAL_SIZE):
    indices = np.asarray(indices, dtype=np.int64)
    groups = frame['pid'].to_numpy()[indices]
    gss = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
    a, b = next(gss.split(indices, frame['label'].to_numpy()[indices], groups))
    return indices[a], indices[b]

class MedianImputer:
    """Equivalent to SimpleImputer(strategy='median', keep_empty_features=True), but fast on wide data."""
    def fit(self, X):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            med = np.nanmedian(X, axis=0)
        self.med_ = np.where(np.isfinite(med), med, 0.0).astype(np.float32)
        return self
    def transform(self, X):
        X = np.array(X, dtype=np.float32, copy=True)
        r, c = np.where(np.isnan(X))
        X[r, c] = self.med_[c]
        return X

def hotdeck_impute(X_fit, X_list, rng):
    """Fill NaNs by sampling observed TRAINING values of the same column (missingness-blind imputation)."""
    outs = [np.array(X, dtype=np.float32, copy=True) for X in X_list]
    for j in range(X_fit.shape[1]):
        col = X_fit[:, j]
        obs = col[~np.isnan(col)]
        for X in outs:
            m = np.isnan(X[:, j])
            if m.any():
                X[m, j] = rng.choice(obs, size=int(m.sum())) if len(obs) else 0.0
    return outs

def _corr_hits(Z, thr, denom, bs=2048):
    p = Z.shape[1]
    hi, hj = [], []
    if DEVICE.type == 'cuda':
        Zt = torch.from_numpy(Z).to(DEVICE)
        for i0 in range(0, p, bs):
            Zi = Zt[:, i0:i0 + bs]
            for j0 in range(i0, p, bs):
                Cb = (Zi.T @ Zt[:, j0:j0 + bs]) / denom
                hit = Cb.abs() > thr
                if i0 == j0:
                    hit = torch.triu(hit, diagonal=1)
                ij = hit.nonzero()
                if len(ij):
                    ij = ij.cpu().numpy()
                    hi.append(ij[:, 0] + i0); hj.append(ij[:, 1] + j0)
        del Zt
        torch.cuda.empty_cache()
    else:
        for i0 in range(0, p, bs):
            Zi = Z[:, i0:i0 + bs]
            for j0 in range(i0, p, bs):
                hit = np.abs((Zi.T @ Z[:, j0:j0 + bs]) / denom) > thr
                if i0 == j0:
                    hit = np.triu(hit, k=1)
                a, b = np.nonzero(hit)
                hi.append(a + i0); hj.append(b + j0)
    if not hi:
        return np.zeros(0, np.int64), np.zeros(0, np.int64)
    return np.concatenate(hi), np.concatenate(hj)

class TrainOnlyClusterReducer:
    """Mean-aggregate clusters of |r| > threshold features (train-only fit).

    Float32 Pearson correlation on standardised training columns, transitive union of pairs above the
    threshold, clusters ordered by first member, representative = highest training-SD member. Correlation
    blocks run on the GPU when available; groups are found as connected components."""
    def __init__(self, corr_threshold=CORR_THRESHOLD, numerical_tolerance=1e-6):
        self.corr_threshold, self.numerical_tolerance = corr_threshold, numerical_tolerance
    def fit(self, X):
        X = np.asarray(X, dtype=np.float32)
        n, p = X.shape
        means = X.mean(axis=0, dtype=np.float64)
        stds = X.std(axis=0, ddof=1, dtype=np.float64)
        safe = np.flatnonzero(np.isfinite(stds) & (stds > 1e-12))
        self.stds_ = np.nan_to_num(stds).astype(np.float32)
        self.n_features_in_ = p
        rows = cols = np.zeros(0, np.int64)
        if len(safe) > 1:
            Z = ((X[:, safe] - means[safe]) / stds[safe]).astype(np.float32)
            Z = np.nan_to_num(Z, copy=False)
            a, b = _corr_hits(Z, float(self.corr_threshold) + float(self.numerical_tolerance), np.float32(max(n - 1, 1)))
            rows, cols = safe[a], safe[b]
            del Z
        G = sp.coo_matrix((np.ones(len(rows), dtype=np.int8), (rows, cols)), shape=(p, p))
        k, lab = connected_components(G, directed=False)
        first = np.full(k, p, dtype=np.int64)
        np.minimum.at(first, lab, np.arange(p))
        rank = np.empty(k, dtype=np.int64)
        rank[np.argsort(first, kind='stable')] = np.arange(k)
        lab = rank[lab]
        order = np.argsort(lab, kind='stable')
        sizes = np.bincount(lab, minlength=k)
        self.clusters_ = np.split(order, np.cumsum(sizes)[:-1])
        self.n_output_features_ = k
        self.avg_ = sp.csr_matrix(((1.0 / sizes[lab]).astype(np.float32), (np.arange(p), lab)), shape=(p, k))
        return self
    def transform(self, X):
        return np.asarray(np.asarray(X, dtype=np.float32) @ self.avg_, dtype=np.float32)
    def fit_transform(self, X):
        return self.fit(X).transform(X)
    def get_cluster_names(self, names):
        return [names[m[int(np.argmax(self.stds_[m]))]] for m in self.clusters_]

# --- Training-only channel filter (fitted per split) ---------------------------------------------------------------
FRAME_PPKEY, _SPLIT_CH, _COLCH = {}, {}, {}
CH_INDEX = {c: i for i, c in enumerate(keep_feature_cols)}

def _frame_ppkey(frame):
    if frame not in FRAME_PPKEY:
        f = FRAMES[frame]
        FRAME_PPKEY[frame] = (f['dataset'].astype(str) + '|' + f['pid'].astype(str)).to_numpy()
    return FRAME_PPKEY[frame]

def channel_keep_mask(frame, rows):
    # Keep raw channels with <= MAX_MISSING_FRAC missing daily values over all daily rows of the participants in
    # `rows` (label-free; callers pass training rows only).
    ix = np.array([PP_INDEX[k] for k in np.unique(_frame_ppkey(frame)[np.asarray(rows)])], dtype=np.int64)
    return PP_NAN[ix].sum(axis=0) / max(int(PP_ROWS[ix].sum()), 1) <= MAX_MISSING_FRAC

def split_channel_mask(split):
    if split not in _SPLIT_CH:
        S = SPLITS[split]
        _SPLIT_CH[split] = channel_keep_mask(S['frame'], S['train_idx'])
    return _SPLIT_CH[split]

def _raw_channel(col):
    c = col[:-len('_selfdev_causal')] if col.endswith('_selfdev_causal') else col
    m = re.match(r'^(.*)_(mean|std|last|missing)$', c)
    return CH_INDEX.get(m.group(1), -1) if m else -1

def col_channels(fs):
    if fs not in _COLCH:
        _COLCH[fs] = np.array([_raw_channel(c) for c in FEATURE_SETS[fs]], dtype=np.int64)
    return _COLCH[fs]

def allowed_columns(fs, keep_ch):
    # Columns of feature set `fs` whose raw channel passes the filter (history/availability columns always pass).
    cc = col_channels(fs)
    return np.flatnonzero((cc < 0) | keep_ch[np.maximum(cc, 0)])

_PREP_CACHE = OrderedDict()
_PREP_CACHE_MAX = 2

def get_prep(split_name, seed, fs, imputation='median', reduce=True):
    # Train-only preprocessing for one (split, seed): channel filter (outer-training participants) -> inner split ->
    # impute -> correlation reduction -> variance filter (all fitted on inner-training rows).
    key = (split_name, seed, fs, imputation, reduce)
    if key in _PREP_CACHE:
        _PREP_CACHE.move_to_end(key)
        return _PREP_CACHE[key]
    S = SPLITS[split_name]
    frame = FRAMES[S['frame']]
    cols = allowed_columns(fs, split_channel_mask(split_name))
    names = [FEATURE_SETS[fs][i] for i in cols]
    XM = feature_matrix(S['frame'], fs)
    tr, va = make_group_inner_split(S['train_idx'], frame, seed)
    te = S['test_idx']
    Xtr0, Xva0, Xte0 = XM[np.ix_(tr, cols)], XM[np.ix_(va, cols)], XM[np.ix_(te, cols)]
    if imputation == 'hotdeck':
        Xtr, Xva, Xte = hotdeck_impute(Xtr0, [Xtr0, Xva0, Xte0], np.random.default_rng(seed))
    else:
        imp = MedianImputer().fit(Xtr0)
        Xtr, Xva, Xte = imp.transform(Xtr0), imp.transform(Xva0), imp.transform(Xte0)
    del Xtr0, Xva0, Xte0
    if reduce:
        red = TrainOnlyClusterReducer().fit(Xtr)
        Xtr, Xva, Xte = red.transform(Xtr), red.transform(Xva), red.transform(Xte)
        rep_names = red.get_cluster_names(names)
        members = [[names[i] for i in m] for m in red.clusters_]
    else:
        rep_names, members = list(names), [[n] for n in names]
    keep = Xtr.var(axis=0) > VARIANCE_THRESHOLD
    y = frame['label'].to_numpy().astype(int)
    yr = frame['year'].to_numpy().astype(int)
    pid = frame['pid'].to_numpy()
    P = dict(split=split_name, seed=seed, tr=tr, va=va, te=te, n_filtered=len(cols),
             Xtr=np.ascontiguousarray(Xtr[:, keep]), Xva=np.ascontiguousarray(Xva[:, keep]),
             Xte=np.ascontiguousarray(Xte[:, keep]),
             ytr=y[tr], yva=y[va], yte=y[te], years_tr=yr[tr], years_va=yr[va], years_te=yr[te],
             pids_te=pid[te], feature_names=[n for n, k in zip(rep_names, keep) if k],
             members=[m for m, k in zip(members, keep) if k])
    _PREP_CACHE[key] = P
    while len(_PREP_CACHE) > _PREP_CACHE_MAX:
        _PREP_CACHE.popitem(last=False)
    return P

def scaled(P):
    if 'Xtr_s' not in P:
        mu = P['Xtr'].mean(axis=0)
        sd = P['Xtr'].std(axis=0)
        sd[sd < 1e-8] = 1.0
        for k in ('Xtr', 'Xva', 'Xte'):
            P[k + '_s'] = ((P[k] - mu) / sd).astype(np.float32)
    return P['Xtr_s'], P['Xva_s'], P['Xte_s']

class SeqStore:
    # All weekly windows (N, 28, C_candidate) as one tensor, on the GPU when it fits. Each split uses its own
    # training-only channel subset. Standardisation statistics come from inner-training windows only; missing values
    # are mean-imputed and an observed-value mask is concatenated.
    def __init__(self, seq):
        X = torch.from_numpy(seq)
        self.on_gpu = False
        if DEVICE.type == 'cuda':
            free, _ = torch.cuda.mem_get_info()
            if X.numel() * 4 < 0.5 * free:
                X = X.to(DEVICE)
                self.on_gpu = True
        self.X = X
        self._stats = {}
        print(f'SeqStore: {tuple(X.shape)} on {"GPU" if self.on_gpu else "CPU"}')
    def channels(self, ch):
        return torch.as_tensor(np.asarray(ch, dtype=np.int64), device=self.X.device)
    def _take(self, idx, ch_t):
        x = self.X[torch.as_tensor(np.asarray(idx, dtype=np.int64), device=self.X.device)]
        return x.index_select(2, ch_t).to(DEVICE)
    def stats(self, idx, ch_t, key=None):
        if key is not None and key in self._stats:
            return self._stats[key]
        s = torch.zeros(len(ch_t), dtype=torch.float64, device=DEVICE)
        sq, cnt = torch.zeros_like(s), torch.zeros_like(s)
        idx = np.asarray(idx)
        for i in range(0, len(idx), 512):
            x = self._take(idx[i:i + 512], ch_t).double()
            v = ~torch.isnan(x)
            x = torch.where(v, x, torch.zeros_like(x))
            s += x.sum((0, 1)); sq += (x * x).sum((0, 1)); cnt += v.sum((0, 1))
        mean = torch.where(cnt > 0, s / cnt.clamp_min(1), torch.zeros_like(s))
        var = torch.where(cnt > 0, sq / cnt.clamp_min(1), torch.zeros_like(s)) - mean ** 2
        out = (mean.float(), torch.sqrt(var.clamp_min(1e-6)).float())
        if key is not None:
            self._stats[key] = out
        return out
    def batch(self, idx, mean, std, ch_t):
        x = self._take(idx, ch_t)
        v = ~torch.isnan(x)
        x = (torch.where(v, x, mean) - mean) / std
        return torch.cat([x, v.float()], dim=2)

def seq_channels_for(split):
    # Raw channels for GRU/Transformer on this split: main-feature channels passing the training-only filter.
    return np.flatnonzero(split_channel_mask(split) & SEQ_MAIN_MASK)

SEQ_STORE = SeqStore(SEQ)
gc.collect()

### Diagnostic: mixed-sign correlation groups
The reducer replaces each group of features with |r| > 0.9 by the mean of its members, so negatively correlated
members partly cancel. For the 2021 forward split (seed 0) this cell counts the multi-member groups that contain a
negatively correlated member and how many of them the variance filter then removes. It does not affect any result.

In [ ]:
S = SPLITS['main|fwd_2021']; frame = FRAMES[S['frame']]
cols = allowed_columns('main', split_channel_mask('main|fwd_2021'))
X0 = feature_matrix(S['frame'], 'main')
tr, va = make_group_inner_split(S['train_idx'], frame, 0)
Xtr = MedianImputer().fit(X0[np.ix_(tr, cols)]).transform(X0[np.ix_(tr, cols)])
red = TrainOnlyClusterReducer().fit(Xtr)
comp = red.transform(Xtr)
multi = mixed = dropped = dropped_mixed = 0
for k, m in enumerate(red.clusters_):
    if len(m) < 2:
        continue
    multi += 1
    rep = m[int(np.argmax(red.stds_[m]))]
    neg = any(np.corrcoef(Xtr[:, rep], Xtr[:, j])[0, 1] < 0 for j in m if j != rep)
    low = comp[:, k].var() <= VARIANCE_THRESHOLD
    mixed += neg; dropped += low; dropped_mixed += (neg and low)
print('multi-member groups:', multi, '| mixed-sign:', mixed,
      '| removed by variance filter:', dropped, '| of which mixed-sign:', dropped_mixed)

### Attribution maps: pooled correlation clusters, behavioural constructs, modalities, feature types
The pooled cluster map is unsupervised (no labels), fitted once on the pooled feature table and used only for
reporting attributions. Constructs strip window, aggregation statistic and `_norm`; credit for a train-split
cluster is shared among its members in proportion to membership (so a cluster spanning two modalities is split).

In [ ]:
def parse_feature(col):
    c = str(col)
    if c.startswith('hist_avail:'):
        return dict(modality=modality_of(c), construct='hist_avail', ftype='availability')
    selfdev = c.endswith('_selfdev_causal')
    if selfdev:
        c = c[:-len('_selfdev_causal')]
    stat = None
    for s in ('_mean', '_std', '_last', '_missing'):
        if c.endswith(s):
            stat, c = s[1:], c[:-len(s)]
            break
    parts = c.split(':')
    mid = re.sub(r'_norm$', '', parts[1]) if len(parts) > 1 else ''
    ftype = 'self-deviation' if selfdev else ('missingness' if stat == 'missing' else 'value')
    return dict(modality=modality_of(parts[0]), construct=f'{parts[0]}:{mid}', ftype=ftype)

def _data_signature():
    # Exact, label-inclusive signature of the analysis data (no float sums, so it is identical across machines).
    h = hashlib.md5()
    for name, f in FRAMES.items():
        h.update(name.encode())
        h.update(pd.util.hash_pandas_object(f[['pid', 'dataset', 'year', 'label_date', 'label', 'valid_days']],
                                            index=False).to_numpy().tobytes())
    for frame, fs in (('ml', 'main'), ('bdi', 'bdi')):
        if frame in FRAMES and FEATURE_SETS[fs]:
            h.update('|'.join(FEATURE_SETS[fs]).encode())
            h.update(np.isfinite(feature_matrix(frame, fs)).sum(axis=0).astype(np.int64).tobytes())
    h.update('|'.join(map(str, PP_KEYS)).encode()); h.update(PP_ROWS.tobytes()); h.update(PP_NAN.tobytes())
    h.update('|'.join(keep_feature_cols).encode())
    return h.hexdigest()

with stage('Fingerprint data and preprocessing'):
    DATA_FP = fp(_data_signature(), read_features, read_weekly_labels, read_endterm_labels, read_platform, _as_binary,
                 _drop_index_like, _find_col, _norm_pid, build_windows, aggregate_windows, causal_selfdev,
                 _is_selfdev_eligible, _window_token, modality_of,
                 dict(SEQUENCE_DAYS=SEQUENCE_DAYS, MIN_VALID_DAYS=MIN_VALID_DAYS, MAX_MISSING_FRAC=MAX_MISSING_FRAC,
                      EXCLUDE=EXCLUDE_POPULATION_NORM_FEATURES, BDI_THRESHOLD=BDI_THRESHOLD, LABEL_COLUMN=LABEL_COLUMN,
                      FEATURE_FILE=FEATURE_FILE, DATASET_YEARS=DATASET_YEARS))
    PREP_FP = fp(DATA_FP, get_prep, make_group_inner_split, MedianImputer, hotdeck_impute, TrainOnlyClusterReducer,
                 _corr_hits, scaled, channel_keep_mask, split_channel_mask, allowed_columns, col_channels, _raw_channel,
                 _frame_ppkey, dict(VARIANCE_THRESHOLD=VARIANCE_THRESHOLD, CORR_THRESHOLD=CORR_THRESHOLD,
                                    INNER_VAL_SIZE=INNER_VAL_SIZE, MAX_MISSING_FRAC=MAX_MISSING_FRAC))
    print('data fingerprint', DATA_FP, '| preprocessing fingerprint', PREP_FP)

def split_fp(split):
    S = SPLITS[split]
    return fp(S['frame'], S['train_idx'], S['test_idx'])

with stage('Pooled attribution cluster map'):
    def _pooled_map():
        X = MedianImputer().fit(feature_matrix('ml', 'main')).transform(feature_matrix('ml', 'main'))
        red = TrainOnlyClusterReducer().fit(X)
        reps = red.get_cluster_names(MAIN_FEATURES)
        return {MAIN_FEATURES[i]: reps[j] for j, m in enumerate(red.clusters_) for i in m}
    CLUSTER_FP = fp(DATA_FP, _pooled_map, MedianImputer, TrainOnlyClusterReducer, _corr_hits, CORR_THRESHOLD)
    feature_rep_of = cached(f'pooled_cluster_map__{CLUSTER_FP}', _pooled_map)

FEATURE_INFO = {c: parse_feature(c) for c in FEATURES_ALL}
LEVELS = ['cluster', 'construct', 'modality', 'ftype']
ITEMS = {'cluster': sorted(set(feature_rep_of.values()) | set(FEATURES_ALL) - set(feature_rep_of)),
         'construct': sorted({v['construct'] for v in FEATURE_INFO.values()}),
         'modality': sorted({v['modality'] for v in FEATURE_INFO.values()}),
         'ftype': sorted({v['ftype'] for v in FEATURE_INFO.values()})}
ITEM_INDEX = {lvl: {it: i for i, it in enumerate(items)} for lvl, items in ITEMS.items()}
print({k: len(v) for k, v in ITEMS.items()})

def attribution_matrices(feature_names, members):
    """Sparse (p_reduced x n_items) credit matrices per level."""
    mats = {}
    p = len(feature_names)
    r = [ITEM_INDEX['cluster'][feature_rep_of.get(n, n)] for n in feature_names]
    mats['cluster'] = sp.csr_matrix((np.ones(p, np.float32), (np.arange(p), r)), shape=(p, len(ITEMS['cluster'])))
    for lvl in ('construct', 'modality', 'ftype'):
        ri, ci, vi = [], [], []
        for j, mem in enumerate(members):
            w = 1.0 / len(mem)
            for m in mem:
                ri.append(j); ci.append(ITEM_INDEX[lvl][FEATURE_INFO[m][lvl]]); vi.append(w)
        mats[lvl] = sp.csr_matrix((np.asarray(vi, np.float32), (ri, ci)), shape=(p, len(ITEMS[lvl])))
    return mats

ATTR_FP = fp(CLUSTER_FP, parse_feature, attribution_matrices)

## 7. Models
All learned transforms are fitted on the inner-training participants of the outer training set; the
inner-validation participants select checkpoints, decision thresholds and (Section 8) hyperparameters.
The outer test set is scored once per (model, split, seed).

In [ ]:
TABULAR = ['Dummy', 'LogReg', 'XGBoost', 'MLP-ERM', 'MLP-GroupDRO', 'MLP-IRM']
SEQUENTIAL = ['GRU', 'Transformer']
MULTI_DOMAIN_ONLY = ['MLP-GroupDRO', 'MLP-IRM']

def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def fit_dummy(P, params, seed, trial=None):
    clf = DummyClassifier(strategy='stratified', random_state=seed).fit(np.zeros((len(P['ytr']), 1)), P['ytr'])
    return dict(pva=clf.predict_proba(np.zeros((len(P['yva']), 1)))[:, 1],
                pte=clf.predict_proba(np.zeros((len(P['yte']), 1)))[:, 1], fixed_threshold=0.5)

def fit_logreg(P, params, seed, trial=None):
    Xtr, Xva, Xte = scaled(P)
    clf = LogisticRegression(C=float(params['C']), solver='lbfgs', max_iter=3000, class_weight='balanced',
                             random_state=seed)
    clf.fit(Xtr, P['ytr'])
    return dict(pva=clf.predict_proba(Xva)[:, 1], pte=clf.predict_proba(Xte)[:, 1])

def make_xgb(params, seed, ytr):
    pos_w = float((ytr == 0).sum()) / max(int((ytr == 1).sum()), 1)
    return xgb.XGBClassifier(n_estimators=int(params['n_estimators']), max_depth=int(params['max_depth']),
                             learning_rate=float(params['learning_rate']), subsample=float(params['subsample']),
                             colsample_bytree=float(params['colsample_bytree']),
                             min_child_weight=float(params['min_child_weight']),
                             reg_lambda=float(params['reg_lambda']), reg_alpha=float(params['reg_alpha']),
                             gamma=float(params['gamma']), scale_pos_weight=pos_w, eval_metric='logloss',
                             random_state=seed, n_jobs=-1, verbosity=0, **xgb_device_kwargs())

def fit_xgb(P, params, seed, trial=None):
    m = make_xgb(params, seed, P['ytr']).fit(P['Xtr'], P['ytr'])
    return dict(pva=m.predict_proba(P['Xva'])[:, 1], pte=m.predict_proba(P['Xte'])[:, 1], model=m)

def make_mlp(d, hidden, n_layers, dropout):
    layers, din = [], d
    for _ in range(int(n_layers)):
        layers += [nn.Linear(din, int(hidden)), nn.ReLU(), nn.Dropout(float(dropout))]
        din = int(hidden)
    layers.append(nn.Linear(din, 2))
    return nn.Sequential(*layers)

class GroupDROLoss:
    def __init__(self, n_domains, eta):
        self.q = torch.ones(n_domains, device=DEVICE) / n_domains
        self.eta = eta
    def __call__(self, logits, yb, db, crit_none):
        per = crit_none(logits, yb)
        present = torch.unique(db)
        gl = torch.stack([per[db == g].mean() for g in present])
        self.q[present] = self.q[present] * torch.exp(self.eta * gl.detach())
        self.q = self.q / self.q.sum()
        qp = self.q[present] / self.q[present].sum()
        return (qp.detach() * gl).sum()

def irm_loss(model, Xb, yb, db, crit_mean, penalty_weight):
    losses, pens = [], []
    for e in torch.unique(db):
        m = db == e
        scale = torch.ones((), device=DEVICE, requires_grad=True)
        le = crit_mean(model(Xb[m]) * scale, yb[m])
        g = torch.autograd.grad(le, [scale], create_graph=True)[0]
        losses.append(le); pens.append(g ** 2)
    loss = torch.stack(losses).mean() + penalty_weight * torch.stack(pens).mean()
    return loss / penalty_weight if penalty_weight > 1.0 else loss

def domain_balanced_batches(dtr, batch_size, rng):
    doms = np.unique(dtr)
    per = max(1, batch_size // len(doms))
    pools = {d: np.flatnonzero(dtr == d) for d in doms}
    for _ in range(max(int(np.ceil(len(v) / per)) for v in pools.values())):
        idx = np.concatenate([rng.choice(pools[d], size=per, replace=len(pools[d]) < per) for d in doms])
        rng.shuffle(idx)
        yield idx

def fit_mlp(P, params, seed, trial=None, method='ERM'):
    set_all_seeds(seed)
    Xtr, Xva, Xte = scaled(P)
    T = lambda a, dt=torch.float32: torch.as_tensor(a, dtype=dt, device=DEVICE)
    Xtr_t, ytr_t, Xva_t, Xte_t = T(Xtr), T(P['ytr'], torch.long), T(Xva), T(Xte)
    dom_map = {y: i for i, y in enumerate(sorted(np.unique(P['years_tr'])))}
    dtr = np.array([dom_map[y] for y in P['years_tr']])
    dtr_t = T(dtr, torch.long)
    counts = np.bincount(P['ytr'], minlength=2)
    cw = T(counts.sum() / (2.0 * np.maximum(counts, 1)))
    crit_mean, crit_none = nn.CrossEntropyLoss(weight=cw), nn.CrossEntropyLoss(weight=cw, reduction='none')
    model = make_mlp(Xtr.shape[1], params['hidden'], params['n_layers'], params['dropout']).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=float(params['lr']), weight_decay=float(params['weight_decay']))
    gdro = GroupDROLoss(len(dom_map), float(params.get('eta', 0.01))) if method == 'GroupDRO' else None
    bs = int(params['batch_size'])
    best, best_state, best_epoch, stale = -np.inf, None, 0, 0
    for epoch in range(1, MLP_MAX_EPOCHS + 1):
        model.train()
        rng = np.random.default_rng(seed * 100003 + epoch)
        if method in ('GroupDRO', 'IRM'):
            batches = domain_balanced_batches(dtr, bs, rng)
        else:
            perm = rng.permutation(len(dtr))
            batches = (perm[i:i + bs] for i in range(0, len(perm), bs))
        for b in batches:
            bi = torch.as_tensor(b, device=DEVICE)
            Xb, yb, db = Xtr_t[bi], ytr_t[bi], dtr_t[bi]
            opt.zero_grad()
            if method == 'ERM':
                loss = crit_mean(model(Xb), yb)
            elif method == 'GroupDRO':
                loss = gdro(model(Xb), yb, db, crit_none)
            else:
                pw = float(params['irm_lambda']) if epoch >= int(params['irm_anneal']) else 1.0
                loss = irm_loss(model, Xb, yb, db, crit_mean, pw)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            pva = torch.softmax(model(Xva_t), 1)[:, 1].cpu().numpy()
        auc = safe_auc(P['yva'], pva)
        score = auc if np.isfinite(auc) else 0.5
        if trial is not None:
            trial.report(score, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
        if score > best + 1e-4:
            best, best_epoch, stale = score, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= MLP_PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pva = torch.softmax(model(Xva_t), 1)[:, 1].cpu().numpy()
        pte = torch.softmax(model(Xte_t), 1)[:, 1].cpu().numpy()
    worst = np.nanmin([safe_auc(P['yva'][P['years_va'] == y], pva[P['years_va'] == y])
                       for y in np.unique(P['years_va'])] + [np.inf])
    return dict(pva=pva, pte=pte, best_epoch=best_epoch, val_worst_domain_auc=float(worst) if np.isfinite(worst) else np.nan)

class GRUDepression(nn.Module):
    def __init__(self, n_value, hidden_size=32, num_layers=1, dropout=0.2):
        super().__init__()
        self.n_value = n_value
        self.gru = nn.GRU(2 * n_value, int(hidden_size), int(num_layers), batch_first=True,
                          dropout=float(dropout) if int(num_layers) > 1 else 0.0)
        self.drop = nn.Dropout(float(dropout))
        self.fc = nn.Linear(int(hidden_size), 2)
    def forward(self, x):
        out, _ = self.gru(x)
        valid = (x[:, :, self.n_value:].sum(2) > 0).float().unsqueeze(-1)
        pooled = (out * valid).sum(1) / valid.sum(1).clamp_min(1.0)
        return self.fc(self.drop(pooled))

class TransformerDepression(nn.Module):
    def __init__(self, n_value, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.2):
        super().__init__()
        self.n_value = n_value
        self.inp = nn.Linear(2 * n_value, int(d_model))
        self.pos = nn.Parameter(torch.zeros(1, SEQUENCE_DAYS, int(d_model)))
        nn.init.trunc_normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(int(d_model), int(nhead), int(dim_feedforward), float(dropout),
                                           batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, int(num_layers), enable_nested_tensor=False)
        self.drop = nn.Dropout(float(dropout))
        self.fc = nn.Linear(int(d_model), 2)
    def forward(self, x):
        valid = x[:, :, self.n_value:].sum(2) > 0
        kpm = ~valid & valid.any(1, keepdim=True)
        h = self.enc(self.inp(x) + self.pos[:, :x.shape[1]], src_key_padding_mask=kpm)
        vf = valid.float().unsqueeze(-1)
        return self.fc(self.drop((h * vf).sum(1) / vf.sum(1).clamp_min(1.0)))

@torch.no_grad()
def seq_predict(model, idx, mean, std, ch_t, bs=256):
    model.eval()
    out = []
    for i in range(0, len(idx), bs):
        out.append(torch.softmax(model(SEQ_STORE.batch(idx[i:i + bs], mean, std, ch_t)), 1)[:, 1].cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)

def fit_seq(split_name, seed, params, arch, trial=None):
    set_all_seeds(seed)
    S = SPLITS[split_name]
    tr, va = make_group_inner_split(S['train_idx'], ml_table, seed)
    te = S['test_idx']
    ch_t = SEQ_STORE.channels(seq_channels_for(split_name))   # training-only channel subset for this split
    mean, std = SEQ_STORE.stats(tr, ch_t, key=(split_name, seed))
    C = len(ch_t)
    if arch == 'GRU':
        model = GRUDepression(C, params['hidden'], params['num_layers'], params['dropout'])
    else:
        model = TransformerDepression(C, params['d_model'], params['nhead'], params['num_layers'],
                                      params['dim_feedforward'], params['dropout'])
    model = model.to(DEVICE)
    ytr = ml_table['label'].to_numpy()[tr].astype(int)
    counts = np.bincount(ytr, minlength=2)
    crit = nn.CrossEntropyLoss(weight=torch.as_tensor(counts.sum() / (2.0 * np.maximum(counts, 1)),
                                                      dtype=torch.float32, device=DEVICE))
    opt = torch.optim.AdamW(model.parameters(), lr=float(params['lr']), weight_decay=float(params['weight_decay']))
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=3, min_lr=1e-5)
    yva = ml_table['label'].to_numpy()[va].astype(int)
    ytr_t = torch.as_tensor(ytr, device=DEVICE)
    bs = int(params['batch_size'])
    g = torch.Generator().manual_seed(seed)
    best, best_state, best_epoch, stale = -np.inf, None, 0, 0
    for epoch in range(1, SEQ_MAX_EPOCHS + 1):
        model.train()
        perm = torch.randperm(len(tr), generator=g).numpy()
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            opt.zero_grad()
            loss = crit(model(SEQ_STORE.batch(tr[b], mean, std, ch_t)), ytr_t[torch.as_tensor(b, device=DEVICE)])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        pva = seq_predict(model, va, mean, std, ch_t)
        auc = safe_auc(yva, pva)
        score = auc if np.isfinite(auc) else 0.5
        sched.step(score)
        if trial is not None:
            trial.report(score, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
        if score > best + 1e-4:
            best, best_epoch, stale = score, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= SEQ_PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    pva, pte = seq_predict(model, va, mean, std, ch_t), seq_predict(model, te, mean, std, ch_t)
    return dict(pva=pva, pte=pte, best_epoch=best_epoch, yva=yva, n_features_filtered=C, n_features_model=C,
                yte=ml_table['label'].to_numpy()[te].astype(int), pids_te=ml_table['pid'].to_numpy()[te],
                years_te=ml_table['year'].to_numpy()[te].astype(int))

FITTERS = {'Dummy': fit_dummy, 'LogReg': fit_logreg, 'XGBoost': fit_xgb,
           'MLP-ERM': lambda P, pr, s, trial=None: fit_mlp(P, pr, s, trial, 'ERM'),
           'MLP-GroupDRO': lambda P, pr, s, trial=None: fit_mlp(P, pr, s, trial, 'GroupDRO'),
           'MLP-IRM': lambda P, pr, s, trial=None: fit_mlp(P, pr, s, trial, 'IRM')}

## 8. Hyperparameter optimisation (Optuna, inner validation only)
One study per (analysis, model, outer split). The objective is ROC AUC on the participant-disjoint inner
validation set of the tuning seed; the outer test year never enters. A fixed default configuration is
enqueued as trial 0. Neural studies use median pruning on per-epoch validation AUC.

In [ ]:
DEFAULT_PARAMS = {
    'Dummy': {},
    'LogReg': {'C': 1.0},
    'XGBoost': dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                    min_child_weight=1.0, reg_lambda=1.0, reg_alpha=1e-4, gamma=0.0),
    'MLP-ERM': dict(hidden=128, n_layers=2, dropout=0.3, lr=1e-3, weight_decay=1e-4, batch_size=256),
    'GRU': dict(hidden=32, num_layers=1, dropout=0.2, lr=5e-4, weight_decay=1e-4, batch_size=64),
    'Transformer': dict(d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.2, lr=5e-4,
                        weight_decay=1e-4, batch_size=64)}
DEFAULT_PARAMS['MLP-GroupDRO'] = dict(DEFAULT_PARAMS['MLP-ERM'], eta=0.01)
DEFAULT_PARAMS['MLP-IRM'] = dict(DEFAULT_PARAMS['MLP-ERM'], irm_lambda=100.0, irm_anneal=20)

SEARCH_SPACE_TEXT = {
    'LogReg': 'C ~ logU[1e-4, 10]; L2, class-balanced, lbfgs',
    'XGBoost': 'n_estimators 100-1000 (step 50); max_depth 2-8; learning_rate logU[0.01, 0.3]; subsample U[0.5, 1]; '
               'colsample_bytree U[0.1, 1]; min_child_weight logU[0.5, 20]; reg_lambda logU[1e-3, 10]; '
               'reg_alpha logU[1e-4, 10]; gamma U[0, 5]; scale_pos_weight = neg/pos (fixed)',
    'MLP': 'hidden {64,128,256,512}; layers 1-3; dropout U[0, 0.6]; lr logU[1e-4, 3e-3]; weight_decay logU[1e-6, 1e-2]; '
           'batch {128,256,512}; max 80 epochs, patience 15 on validation AUC',
    'MLP-GroupDRO': '+ eta logU[1e-3, 1]',
    'MLP-IRM': '+ penalty weight logU[1, 1e4]; anneal epoch 0-40',
    'GRU': 'hidden {16,32,64,128}; layers 1-2; dropout U[0, 0.5]; lr logU[1e-4, 3e-3]; weight_decay logU[1e-6, 1e-2]; '
           'batch {32,64,128}; max 60 epochs, patience 10',
    'Transformer': 'd_model {32,64,128}; heads {2,4,8}; layers 1-3; feed-forward {64,128,256}; dropout U[0, 0.5]; '
                   'lr logU[1e-4, 3e-3]; weight_decay logU[1e-6, 1e-2]; batch {32,64,128}'}

def suggest(model, t):
    if model == 'LogReg':
        return {'C': t.suggest_float('C', 1e-4, 10.0, log=True)}
    if model == 'XGBoost':
        return dict(n_estimators=t.suggest_int('n_estimators', 100, 1000, step=50),
                    max_depth=t.suggest_int('max_depth', 2, 8),
                    learning_rate=t.suggest_float('learning_rate', 0.01, 0.3, log=True),
                    subsample=t.suggest_float('subsample', 0.5, 1.0),
                    colsample_bytree=t.suggest_float('colsample_bytree', 0.1, 1.0),
                    min_child_weight=t.suggest_float('min_child_weight', 0.5, 20.0, log=True),
                    reg_lambda=t.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
                    reg_alpha=t.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
                    gamma=t.suggest_float('gamma', 0.0, 5.0))
    if model.startswith('MLP'):
        p = dict(hidden=t.suggest_categorical('hidden', [64, 128, 256, 512]),
                 n_layers=t.suggest_int('n_layers', 1, 3), dropout=t.suggest_float('dropout', 0.0, 0.6),
                 lr=t.suggest_float('lr', 1e-4, 3e-3, log=True),
                 weight_decay=t.suggest_float('weight_decay', 1e-6, 1e-2, log=True),
                 batch_size=t.suggest_categorical('batch_size', [128, 256, 512]))
        if model == 'MLP-GroupDRO':
            p['eta'] = t.suggest_float('eta', 1e-3, 1.0, log=True)
        if model == 'MLP-IRM':
            p['irm_lambda'] = t.suggest_float('irm_lambda', 1.0, 1e4, log=True)
            p['irm_anneal'] = t.suggest_int('irm_anneal', 0, 40)
        return p
    if model == 'GRU':
        return dict(hidden=t.suggest_categorical('hidden', [16, 32, 64, 128]), num_layers=t.suggest_int('num_layers', 1, 2),
                    dropout=t.suggest_float('dropout', 0.0, 0.5), lr=t.suggest_float('lr', 1e-4, 3e-3, log=True),
                    weight_decay=t.suggest_float('weight_decay', 1e-6, 1e-2, log=True),
                    batch_size=t.suggest_categorical('batch_size', [32, 64, 128]))
    if model == 'Transformer':
        return dict(d_model=t.suggest_categorical('d_model', [32, 64, 128]), nhead=t.suggest_categorical('nhead', [2, 4, 8]),
                    num_layers=t.suggest_int('num_layers', 1, 3),
                    dim_feedforward=t.suggest_categorical('dim_feedforward', [64, 128, 256]),
                    dropout=t.suggest_float('dropout', 0.0, 0.5), lr=t.suggest_float('lr', 1e-4, 3e-3, log=True),
                    weight_decay=t.suggest_float('weight_decay', 1e-6, 1e-2, log=True),
                    batch_size=t.suggest_categorical('batch_size', [32, 64, 128]))
    return {}

def run_fit(model, split, seed, params, fs, trial=None):
    if model in SEQUENTIAL:
        return fit_seq(split, seed, params, model, trial)
    P = get_prep(split, seed, fs)
    out = FITTERS[model](P, params, seed, trial)
    out.update(yva=P['yva'], yte=P['yte'], pids_te=P['pids_te'], years_te=P['years_te'],
               n_features_filtered=P['n_filtered'], n_features_model=P['Xtr'].shape[1])
    return out

MODEL_CODE = {'Dummy': [fit_dummy], 'LogReg': [fit_logreg, scaled], 'XGBoost': [fit_xgb, make_xgb, xgb_device_kwargs],
              'MLP-ERM': [fit_mlp, make_mlp, scaled, set_all_seeds],
              'MLP-GroupDRO': [fit_mlp, make_mlp, scaled, set_all_seeds, GroupDROLoss, domain_balanced_batches],
              'MLP-IRM': [fit_mlp, make_mlp, scaled, set_all_seeds, irm_loss, domain_balanced_batches],
              'GRU': [fit_seq, seq_predict, set_all_seeds, GRUDepression, SeqStore, seq_channels_for],
              'Transformer': [fit_seq, seq_predict, set_all_seeds, TransformerDepression, SeqStore, seq_channels_for]}

def model_fp(model):
    return fp(model, *MODEL_CODE[model], FITTERS.get(model), DEFAULT_PARAMS[model], safe_auc,
              dict(mlp=(MLP_MAX_EPOCHS, MLP_PATIENCE), seq=(SEQ_MAX_EPOCHS, SEQ_PATIENCE)))

def hpo_fp(analysis, model, split):
    return fp('hpo', PREP_FP, split_fp(split), ANALYSES[analysis]['feature_set'], model_fp(model), tune, suggest, run_fit,
              dict(TUNING_SEED=TUNING_SEED, trials=HPO_TRIALS.get(model, 0), timeout=HPO_TIMEOUT_S))

HPO = {}   # (analysis, model, split) -> study summary

def tune(analysis, model, split):
    k = (analysis, model, split)
    if k in HPO:
        return HPO[k]
    key = hpo_fp(analysis, model, split)
    name = f'hpo__{analysis}__{model}__{split}__{key}'
    if ck_exists(name):
        res = ck_load(name)
    else:
        n_trials = HPO_TRIALS.get(model, 0)
        if model == 'Dummy' or n_trials <= 0:
            res = dict(best_params=dict(DEFAULT_PARAMS[model]), best_value=np.nan, default_value=np.nan,
                       n_trials=0, n_pruned=0, trials=pd.DataFrame())
        else:
            fs = ANALYSES[analysis]['feature_set']
            sampler = optuna.samplers.TPESampler(seed=stable_seed('hpo', analysis, model, split) % (2 ** 31),
                                                 n_startup_trials=max(5, n_trials // 4))
            pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)
            study = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
            study.enqueue_trial(DEFAULT_PARAMS[model])

            def objective(t):
                out = run_fit(model, split, TUNING_SEED, suggest(model, t), fs, trial=t)
                v = safe_auc(out['yva'], out['pva'])
                if 'val_worst_domain_auc' in out:
                    t.set_user_attr('val_worst_domain_auc', out['val_worst_domain_auc'])
                return v if np.isfinite(v) else 0.5

            t0 = time.time()
            study.optimize(objective, n_trials=n_trials, timeout=HPO_TIMEOUT_S, gc_after_trial=True,
                           catch=(RuntimeError, ValueError))
            done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
            if done:
                best = max(done, key=lambda t: t.value)
                bp = dict(DEFAULT_PARAMS[model]); bp.update(best.params)
                bv = best.value
            else:
                bp, bv = dict(DEFAULT_PARAMS[model]), np.nan
            t0v = study.trials[0].value if study.trials and study.trials[0].state == optuna.trial.TrialState.COMPLETE else np.nan
            tdf = study.trials_dataframe(attrs=('number', 'value', 'state', 'params'))
            res = dict(best_params=bp, best_value=bv, default_value=t0v, n_trials=len(study.trials),
                       n_pruned=sum(t.state == optuna.trial.TrialState.PRUNED for t in study.trials),
                       trials=tdf, seconds=round(time.time() - t0, 1))
            print(f'    HPO {model:12s} {split:22s} default {t0v:.3f} -> best {bv:.3f} '
                  f'({res["n_trials"]} trials, {res["n_pruned"]} pruned, {res["seconds"]:.0f}s)')
        ck_save(name, res)
    res['fp'] = key
    HPO[k] = res
    return res

## 9. Multi-seed training (all analyses)
For each (analysis, split, model): tune once, then train with every seed (each seed has its own inner
split, initialisation and threshold). Results, per-window test predictions and XGBoost SHAP summaries
(per participant, at every attribution level) are checkpointed per (analysis, model, split).

In [ ]:
PARAMS_USED = {}
FINAL_FP = {}
FINAL = {}   # (analysis, model, split) -> dict(rows, preds, shap)

def shap_summary(booster, P):
    contrib = booster.predict(xgb.DMatrix(P['Xte']), pred_contribs=True)[:, :-1].astype(np.float32)
    absC = np.abs(contrib)
    mats = attribution_matrices(P['feature_names'], P['members'])
    codes, uniq = pd.factorize(P['pids_te'])
    M = sp.csr_matrix((np.ones(len(codes), np.float32), (codes, np.arange(len(codes)))), shape=(len(uniq), len(codes)))
    pid_year = pd.Series(P['years_te']).groupby(codes).first().to_numpy()
    levels, present, signed = {}, {}, {}
    for lvl, A in mats.items():
        lv = np.asarray((A.T @ absC.T).T, dtype=np.float32)
        levels[lvl] = np.asarray(M @ lv, dtype=np.float32)
        present[lvl] = np.asarray(A.sum(axis=0)).ravel() > 0
        signed[lvl] = np.asarray(A.T @ contrib.mean(axis=0)).ravel().astype(np.float32)
    feat = pd.DataFrame({'feature': P['feature_names'], 'mean_abs_shap': absC.mean(0), 'mean_signed_shap': contrib.mean(0),
                         'n_members': [len(m) for m in P['members']]})
    return dict(pids=np.asarray(uniq), pid_year=pid_year, pid_rows=np.bincount(codes).astype(np.float32),
                levels=levels, present=present, signed=signed, feat=feat)

def _record(analysis, model, split, seed, out, params):
    thr = out.get('fixed_threshold', None)
    if thr is None:
        thr = choose_validation_threshold(out['yva'], out['pva'])
    pred = (out['pte'] >= thr).astype(int)
    m = metric_dict(out['yte'], pred, out['pte'])
    row = dict(analysis=analysis, model=model, split=split, seed=seed, **m, decision_threshold=thr,
               val_auc=safe_auc(out['yva'], out['pva']), best_epoch=out.get('best_epoch', np.nan),
               val_worst_domain_auc=out.get('val_worst_domain_auc', np.nan),
               n_features_filtered=out.get('n_features_filtered', np.nan), n_features_model=out.get('n_features_model', np.nan),
               collapsed=bool(m['pred_positive_rate'] < 0.05 or m['pred_positive_rate'] > 0.95 or m['prob_std'] < 0.01))
    pdf = pd.DataFrame({'pid': out['pids_te'], 'year': np.asarray(out['years_te'], dtype=np.int16),
                        'y_true': np.asarray(out['yte'], dtype=np.int8), 'proba': np.asarray(out['pte'], dtype=np.float32),
                        'pred': pred.astype(np.int8)})
    return row, pdf

def models_for(analysis, split):
    n_dom = len(SPLITS[split]['train_years'])
    ms = [m for m in ANALYSES[analysis]['models'] if not (m in MULTI_DOMAIN_ONLY and n_dom < 2)]
    if SPLITS[split]['frame'] != 'ml':
        ms = [m for m in ms if m not in SEQUENTIAL]
    return ms

def final_fp(analysis, model, split, param_fp, want_shap):
    shap_part = (shap_summary, attribution_matrices, ATTR_FP) if (want_shap and model == 'XGBoost') else ()
    return fp('final', PREP_FP, split_fp(split), ANALYSES[analysis]['feature_set'], model_fp(model), param_fp, SEEDS,
              _record, metric_dict, choose_validation_threshold, run_fit, *shap_part)

def run_analysis(analysis):
    cfg = ANALYSES[analysis]
    fs = cfg['feature_set']
    want_shap = cfg['shap'] and RUN_EXPLANATION_STABILITY
    for split in splits_of(analysis):
        ms = models_for(analysis, split)
        print(f'[{analysis}] {split}: {ms}')
        params, pfp = {}, {}
        src = cfg['params_from']
        for m in ms:
            if cfg['tune']:
                r = tune(analysis, m, split)
            else:
                sp_src = split.replace(f'{analysis}|', f'{src}|', 1)
                r = tune(src, m, sp_src) if (sp_src in SPLITS and m in models_for(src, sp_src)) else None
            params[m] = dict(r['best_params']) if r else dict(DEFAULT_PARAMS[m])
            pfp[m] = r['fp'] if r else fp('default', DEFAULT_PARAMS[m])
        PARAMS_USED[(analysis, split)] = params
        names = {}
        for m in ms:
            FINAL_FP[(analysis, m, split)] = final_fp(analysis, m, split, pfp[m], want_shap)
            names[m] = f'final__{analysis}__{m}__{split}__{FINAL_FP[(analysis, m, split)]}'
            if ck_exists(names[m]):
                FINAL[(analysis, m, split)] = ck_load(names[m])
        tab = [m for m in ms if m in TABULAR and (analysis, m, split) not in FINAL]
        if tab:
            buf = {m: dict(rows=[], preds={}, shap={}) for m in tab}
            for seed in SEEDS:
                for m in tab:
                    out = run_fit(m, split, seed, params[m], fs)
                    row, pdf = _record(analysis, m, split, seed, out, params[m])
                    buf[m]['rows'].append(row); buf[m]['preds'][seed] = pdf
                    if m == 'XGBoost' and want_shap:
                        buf[m]['shap'][seed] = shap_summary(out['model'].get_booster(), get_prep(split, seed, fs))
                gc.collect()
            for m in tab:
                ck_save(names[m], buf[m])
                FINAL[(analysis, m, split)] = buf[m]
        for m in [m for m in ms if m in SEQUENTIAL and (analysis, m, split) not in FINAL]:
            buf = dict(rows=[], preds={}, shap={})
            for seed in SEEDS:
                out = run_fit(m, split, seed, params[m], fs)
                row, pdf = _record(analysis, m, split, seed, out, params[m])
                buf['rows'].append(row); buf['preds'][seed] = pdf
            ck_save(names[m], buf)
            FINAL[(analysis, m, split)] = buf
        for m in ms:
            r = pd.DataFrame(FINAL[(analysis, m, split)]['rows'])
            print(f'    {m:12s} AUC {r["roc_auc"].mean():.3f} +/- {r["roc_auc"].std():.3f} (n={len(r)})')

for _a in ANALYSES:
    with stage(f'Train analysis: {_a}'):
        run_analysis(_a)

RESULTS = pd.DataFrame([r for v in FINAL.values() for r in v['rows']])
PRED = {(a, m, s, seed): df for (a, m, s), v in FINAL.items() for seed, df in v['preds'].items()}
SHAP_STORE = {(a, s, seed): sh for (a, m, s), v in FINAL.items() if m == 'XGBoost' for seed, sh in v['shap'].items()}
print('Result rows:', len(RESULTS), '| prediction sets:', len(PRED), '| SHAP summaries:', len(SHAP_STORE))

## 10. Uncertainty from participants *and* training (two-level bootstrap)
Each replicate (i) resamples test participants with replacement, keeping all of a participant's windows
together, and (ii) draws one of the trained seeds at random. The resulting interval therefore includes
both participant-sampling and model-training variation. Replicates for the same test set share the same
participant resamples, so model-vs-model (and main-vs-sensitivity) differences are paired; seeds are drawn
independently per model. Pooled forward-chaining estimates average the per-year AUCs within each
replicate. Two-sided bootstrap p-values are Holm-adjusted within each comparison family.
Pooled estimates, model rankings, PPV and comparison families use only the test years shared by every model
(GroupDRO and IRM need two training years, so Forward -> first year is not pooled); per-year results are
reported separately, and all-years pooling per model is given as a secondary column.

In [ ]:
_BOOT_COUNTS = {}

def test_key(split):
    S = SPLITS[split]
    return (S['frame'], hashlib.md5(S['test_idx'].tobytes()).hexdigest())

def boot_counts(split):
    key = test_key(split)
    if key not in _BOOT_COUNTS:
        S = SPLITS[split]
        codes, uniq = pd.factorize(FRAMES[S['frame']]['pid'].to_numpy()[S['test_idx']])
        rng = np.random.default_rng(stable_seed('W', *key))
        C = rng.multinomial(len(uniq), np.full(len(uniq), 1.0 / len(uniq)), size=N_BOOT).astype(np.float32)
        _BOOT_COUNTS[key] = (C, codes)
    return _BOOT_COUNTS[key]

def weighted_auc_batch(y, p, W):
    """AUC (ties count 1/2) for every row of the weight matrix W (B x n)."""
    order = np.argsort(p, kind='mergesort')
    ps, ys = p[order], y[order]
    starts = np.r_[0, np.flatnonzero(np.diff(ps)) + 1]
    Ws = W[:, order]
    gp = np.add.reduceat(Ws * ys, starts, axis=1)
    gn = np.add.reduceat(Ws * (1 - ys), starts, axis=1)
    below = np.cumsum(gn, axis=1) - gn
    num = (gp * (below + 0.5 * gn)).sum(1)
    den = gp.sum(1) * gn.sum(1)
    with np.errstate(all='ignore'):
        return np.where(den > 0, num / den, np.nan)

def weighted_rates(y, h, W):
    with np.errstate(all='ignore'):
        sens = (W * (y * h)).sum(1) / (W * y).sum(1)
        spec = (W * ((1 - y) * (1 - h))).sum(1) / (W * (1 - y)).sum(1)
    return sens, spec

BOOT = {}

def boot_unit(analysis, model, split):
    key = (analysis, model, split)
    if key in BOOT:
        return BOOT[key]
    seeds = sorted(s for (a, m, sp_, s) in PRED if a == analysis and m == model and sp_ == split)
    if not seeds:
        return None
    dfs = [PRED[(analysis, model, split, s)] for s in seeds]
    y = dfs[0]['y_true'].to_numpy().astype(np.float64)
    Pm = np.stack([d['proba'].to_numpy(np.float64) for d in dfs])
    Hm = np.stack([d['pred'].to_numpy(np.float64) for d in dfs])
    C, codes = boot_counts(split)
    assert len(codes) == len(y), 'prediction rows do not match the split test set'
    W = C[:, codes].astype(np.float64)
    rng = np.random.default_rng(stable_seed('S', analysis, model, split))
    sd = rng.integers(len(seeds), size=N_BOOT)
    auc_b, sens_b, spec_b = (np.full(N_BOOT, np.nan) for _ in range(3))
    part_sd = []
    for i in range(len(seeds)):
        a_all = weighted_auc_batch(y, Pm[i], W)
        part_sd.append(np.nanstd(a_all))
        sel = sd == i
        if sel.any():
            auc_b[sel] = a_all[sel]
            sens_b[sel], spec_b[sel] = weighted_rates(y, Hm[i], W[sel])
    seed_auc = np.array([safe_auc(y, Pm[i]) for i in range(len(seeds))])
    seed_sens = np.array([(Hm[i] * y).sum() / max(y.sum(), 1) for i in range(len(seeds))])
    seed_spec = np.array([((1 - Hm[i]) * (1 - y)).sum() / max((1 - y).sum(), 1) for i in range(len(seeds))])
    seed_prauc = np.array([average_precision_score(y, Pm[i]) if 0 < y.mean() < 1 else np.nan for i in range(len(seeds))])
    res = dict(seeds=seeds, n=len(y), n_participants=C.shape[1], prevalence=float(y.mean()),
               auc=float(np.nanmean(seed_auc)), auc_seed=seed_auc, auc_b=auc_b,
               sens=float(seed_sens.mean()), spec=float(seed_spec.mean()), sens_b=sens_b, spec_b=spec_b,
               ba=float(0.5 * (seed_sens + seed_spec).mean()), ba_b=0.5 * (sens_b + spec_b),
               pr_auc=float(np.nanmean(seed_prauc)),
               sd_seed=float(np.nanstd(seed_auc, ddof=1)) if len(seeds) > 1 else np.nan,
               sd_participant=float(np.nanmean(part_sd)), sd_total=float(np.nanstd(auc_b)))
    BOOT[key] = res
    return res

def ci(v):
    v = np.asarray(v)[np.isfinite(v)]
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if len(v) else (np.nan, np.nan)

def fmt_ci(p, lo, hi, dp=3):
    return '' if not np.isfinite(p) else f'{p:.{dp}f} [{lo:.{dp}f}, {hi:.{dp}f}]'

def models_in(analysis):
    return list(dict.fromkeys(m for m in MODELS_ALL + ANALYSES.get(analysis, {}).get('models', [])
                              if any(k[0] == analysis and k[1] == m for k in FINAL)))

def view_table(analysis):
    rows = []
    for v in views_of(analysis):
        for m in dict.fromkeys(models_in(analysis)):
            b = boot_unit(analysis, m, v['split'])
            if b is None:
                continue
            lo, hi = ci(b['auc_b']); blo, bhi = ci(b['ba_b'])
            rows.append(dict(Analysis=analysis, Protocol=v['protocol'], View=v['view'], test_year=v['test_year'],
                             split=v['split'], Model=m, Seeds=len(b['seeds']), Participants=b['n_participants'],
                             Windows=b['n'], Prevalence=b['prevalence'], AUC=b['auc'], AUC_lo=lo, AUC_hi=hi,
                             SD_seed=b['sd_seed'], SD_participant=b['sd_participant'], SD_total=b['sd_total'],
                             PR_AUC=b['pr_auc'], BA=b['ba'], BA_lo=blo, BA_hi=bhi, Sensitivity=b['sens'], Specificity=b['spec']))
    return pd.DataFrame(rows)

def pooled(analysis, protocol, model, years=None, metric='auc'):
    units = []
    for v in views_of(analysis, protocol):
        if years is not None and v['test_year'] not in years:
            continue
        u = boot_unit(analysis, model, v['split'])
        if u is not None:
            units.append((v, u))
    if not units:
        return None
    vec = np.mean([u[f'{metric}_b'] for _, u in units], axis=0)
    point = float(np.mean([u[metric] for _, u in units]))
    lo, hi = ci(vec)
    return dict(point=point, lo=lo, hi=hi, vec=vec, years=[v['test_year'] for v, _ in units], n_views=len(units))

def model_years(analysis, protocol, model):
    return [v['test_year'] for v in views_of(analysis, protocol) if boot_unit(analysis, model, v['split']) is not None]

def common_years(analysis, protocol, models=None):
    # Test years available for every model of the analysis (identical test sets for all comparisons).
    sets = [set(model_years(analysis, protocol, m)) for m in (models or models_in(analysis))]
    sets = [x for x in sets if x]
    return sorted(set.intersection(*sets), key=lambda y: (y is None, y if y is not None else 0)) if sets else []

def _yrs(ys):
    return ', '.join(str(y) for y in ys if y is not None)

def paired(an_a, model_a, an_b, model_b, protocol, years=None, metric='auc', view=None):
    va = {v['view']: v for v in views_of(an_a, protocol) if (years is None or v['test_year'] in years)
          and (view is None or v['view'] == view)}
    vb = {v['view']: v for v in views_of(an_b, protocol) if (years is None or v['test_year'] in years)}
    diffs, pts, used = [], [], []
    for k, v in va.items():
        if k not in vb or test_key(v['split']) != test_key(vb[k]['split']):
            continue
        ua, ub = boot_unit(an_a, model_a, v['split']), boot_unit(an_b, model_b, vb[k]['split'])
        if ua is None or ub is None:
            continue
        diffs.append(ua[f'{metric}_b'] - ub[f'{metric}_b']); pts.append(ua[metric] - ub[metric]); used.append(v['test_year'])
    if not diffs:
        return None
    d = np.mean(diffs, axis=0)
    d = d[np.isfinite(d)]
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    p = float(min(max(p, 1.0 / max(len(d), 1)), 1.0))
    lo, hi = ci(d)
    return dict(delta=float(np.mean(pts)), lo=lo, hi=hi, p=p, years=used,
                equivalent=bool(lo > -EQUIV_MARGIN and hi < EQUIV_MARGIN))

def holm(pvals):
    p = np.asarray(pvals, dtype=float)
    adj = np.full(len(p), np.nan)
    ok = np.flatnonzero(np.isfinite(p))
    running = 0.0
    for rank, i in enumerate(ok[np.argsort(p[ok])]):
        running = max(running, min(1.0, (len(ok) - rank) * p[i]))
        adj[i] = running
    return adj

def pooled_table(analysis):
    # Primary: pooled over the test years shared by every model. Secondary: all years available to each model.
    rows = []
    for proto in (PROTO_FWD, PROTO_LOYO, PROTO_PD):
        cy = common_years(analysis, proto)
        for m in models_in(analysis):
            r = pooled(analysis, proto, m, years=cy)
            if r is None:
                continue
            rb = pooled(analysis, proto, m, years=cy, metric='ba')
            ra = pooled(analysis, proto, m)
            rows.append(dict(Analysis=analysis, Protocol=proto, Model=m, Years=_yrs(r['years']),
                             AUC=r['point'], AUC_lo=r['lo'], AUC_hi=r['hi'], BA=rb['point'], BA_lo=rb['lo'], BA_hi=rb['hi'],
                             **{'AUC (95% CI)': fmt_ci(r['point'], r['lo'], r['hi']),
                                'Years (all available)': _yrs(ra['years']),
                                'AUC all available years (95% CI)': fmt_ci(ra['point'], ra['lo'], ra['hi'])}))
    return pd.DataFrame(rows)

def comparison_table(analysis, ref='XGBoost'):
    rows = []
    for proto in (PROTO_FWD, PROTO_LOYO, PROTO_PD):
        cy = common_years(analysis, proto)
        fams = {f'vs {ref}': [(m, ref) for m in models_in(analysis) if m != ref],
                'DG vs ERM': [(m, 'MLP-ERM') for m in MULTI_DOMAIN_ONLY if m in models_in(analysis)]}
        for fam, pairs in fams.items():
            block = []
            for a, b in pairs:
                r = paired(analysis, a, analysis, b, proto, years=cy)
                if r is not None:
                    block.append(dict(Analysis=analysis, Protocol=proto, Family=fam, Comparison=f'{a} - {b}',
                                      Years=', '.join(str(y) for y in r['years'] if y), Delta_AUC=r['delta'],
                                      CI_lo=r['lo'], CI_hi=r['hi'], p=r['p'], Equivalent=r['equivalent']))
            if block:
                adj = holm([b_['p'] for b_ in block])
                for b_, pa in zip(block, adj):
                    b_['p_Holm'] = pa
                    b_['Significant (Holm)'] = bool(pa < ALPHA)
                rows += block
    return pd.DataFrame(rows)

def per_view_comparisons(analysis, ref='XGBoost'):
    rows = []
    for v in views_of(analysis):
        block = []
        for m in models_in(analysis):
            if m == ref:
                continue
            r = paired(analysis, m, analysis, ref, v['protocol'], view=v['view'])
            if r is not None:
                block.append(dict(Analysis=analysis, Protocol=v['protocol'], View=v['view'], Comparison=f'{m} - {ref}',
                                  Delta_AUC=r['delta'], CI_lo=r['lo'], CI_hi=r['hi'], p=r['p'], Equivalent=r['equivalent']))
        for b_, pa in zip(block, holm([b_['p'] for b_ in block])):
            b_['p_Holm'] = pa
        rows += block
    return pd.DataFrame(rows)

with stage('Two-level bootstrap and comparisons'):
    VIEW_TABLES = {a: view_table(a) for a in ANALYSES}
    POOLED_TABLES = {a: pooled_table(a) for a in ANALYSES}
    COMPARISONS = {a: comparison_table(a) for a in ANALYSES if a != 'normabl'}
    PER_VIEW_COMPARISONS = {a: per_view_comparisons(a) for a in ANALYSES if a != 'normabl'}

def protocol_ladder_table(analysis='main'):
    # Same test participants and the same participant bootstrap resamples: leave-one-year-out (later year(s) added to
    # training) minus forward-chaining (earlier years only). The difference combines the effect of later-year data and
    # of a larger training set. The latest year is skipped (its forward and leave-one-year-out splits are identical).
    fwd = {v['test_year']: v['split'] for v in views_of(analysis, PROTO_FWD)}
    loyo = {v['test_year']: v['split'] for v in views_of(analysis, PROTO_LOYO)}
    rows = []
    for y in sorted(set(fwd) & set(loyo)):
        if fwd[y] == loyo[y] or test_key(fwd[y]) != test_key(loyo[y]):
            continue
        for m in models_in(analysis):
            if m == 'Dummy':
                continue
            ua, ub = boot_unit(analysis, m, loyo[y]), boot_unit(analysis, m, fwd[y])
            if ua is None or ub is None:
                continue
            d = ua['auc_b'] - ub['auc_b']
            d = d[np.isfinite(d)]
            p = 2 * min((d <= 0).mean(), (d >= 0).mean())
            p = float(min(max(p, 1.0 / max(len(d), 1)), 1.0))
            lo, hi = ci(d)
            rows.append({'Test year': y, 'Model': m,
                         'Forward train years': ', '.join(map(str, SPLITS[fwd[y]]['train_years'])),
                         'LOYO train years': ', '.join(map(str, SPLITS[loyo[y]]['train_years'])),
                         'Forward AUC': ub['auc'], 'LOYO AUC': ua['auc'], 'Delta (LOYO - forward)': ua['auc'] - ub['auc'],
                         'CI_lo': lo, 'CI_hi': hi, 'p': p})
    df = pd.DataFrame(rows)
    if len(df):
        df['p_Holm'] = holm(df['p'].to_numpy())
    return df

with stage('Paired leave-one-year-out vs forward-chaining (same test years)'):
    PROTOCOL_LADDER = protocol_ladder_table('main')
if len(PROTOCOL_LADDER):
    display(PROTOCOL_LADDER.round(4))
    KEY_NUMBERS['loyo_minus_forward_by_year'] = {
        int(y): dict(mean_delta=round(float(g['Delta (LOYO - forward)'].mean()), 4),
                     models_positive=int((g['Delta (LOYO - forward)'] > 0).sum()), models=int(len(g)),
                     significant_holm=int((g['p_Holm'] < ALPHA).sum()))
        for y, g in PROTOCOL_LADDER.groupby('Test year')}

display(POOLED_TABLES['main'][['Protocol', 'Model', 'Years', 'AUC (95% CI)']])
display(COMPARISONS['main'])

_fp = POOLED_TABLES['main'][POOLED_TABLES['main']['Protocol'] == PROTO_FWD].sort_values('AUC', ascending=False)
if len(_fp):
    KEY_NUMBERS['forward_pooled_test_years'] = _fp.iloc[0]['Years']
    KEY_NUMBERS['forward_best_model'] = _fp.iloc[0]['Model']
    KEY_NUMBERS['forward_best_auc'] = [round(_fp.iloc[0][k], 4) for k in ('AUC', 'AUC_lo', 'AUC_hi')]
    KEY_NUMBERS['forward_auc_by_model'] = {r['Model']: [round(r['AUC'], 4), round(r['AUC_lo'], 4), round(r['AUC_hi'], 4)]
                                           for _, r in _fp.iterrows()}
_cm = COMPARISONS['main']
if len(_cm):
    _f = _cm[_cm['Protocol'] == PROTO_FWD]
    KEY_NUMBERS['forward_significant_vs_xgb_after_holm'] = int(_f[_f['Family'] == 'vs XGBoost']['Significant (Holm)'].sum())
    KEY_NUMBERS['forward_comparisons_vs_xgb'] = int((_f['Family'] == 'vs XGBoost').sum())
    KEY_NUMBERS['forward_dg_vs_erm'] = {r['Comparison']: [round(r['Delta_AUC'], 4), round(r['CI_lo'], 4), round(r['CI_hi'], 4),
                                                         round(r['p_Holm'], 4)] for _, r in _f[_f['Family'] == 'DG vs ERM'].iterrows()}

## 11. Calibration and positive predictive value at deployment prevalence

In [ ]:
def calibration_metrics(y, p, n_bins=10):
    y = np.asarray(y).astype(int)
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    out = dict(slope=np.nan, citl=np.nan, ece=np.nan, brier=float(np.mean((p - y) ** 2)),
               mean_pred=float(p.mean()), observed=float(y.mean()))
    if len(np.unique(y)) < 2:
        return out
    lp = np.log(p / (1 - p))
    try:
        out['slope'] = float(LogisticRegression(C=1e6, max_iter=1000).fit(lp.reshape(-1, 1), y).coef_[0][0])
    except Exception:
        pass
    nll = lambda a: float(np.mean(np.logaddexp(0, a + lp) - y * (a + lp)))
    out['citl'] = float(minimize_scalar(nll, bounds=(-10, 10), method='bounded').x)
    idx = np.clip(np.digitize(p, np.linspace(0, 1, n_bins + 1)[1:-1]), 0, n_bins - 1)
    out['ece'] = float(sum((idx == b).mean() * abs(p[idx == b].mean() - y[idx == b].mean())
                           for b in range(n_bins) if (idx == b).any()))
    return out

with stage('Calibration'):
    calibration_table = pd.DataFrame([dict(analysis=a, model=m, split=s, seed=seed,
                                           **calibration_metrics(df['y_true'], df['proba']))
                                      for (a, m, s, seed), df in PRED.items() if a in ('main', 'ex2018', 'bdi')])
    _vmap = pd.DataFrame(VIEWS)[['analysis', 'protocol', 'view', 'split', 'test_year']]
    cal_views = calibration_table.merge(_vmap, on=['analysis', 'split'])
    _cy = {(a, pr): set(common_years(a, pr)) for a in ('main', 'ex2018', 'bdi') if a in ANALYSES
           for pr in (PROTO_FWD, PROTO_LOYO, PROTO_PD)}
    cal_views = cal_views[[pr == PROTO_PD or ty in _cy.get((a, pr), set())
                           for a, pr, ty in zip(cal_views['analysis'], cal_views['protocol'], cal_views['test_year'])]]
    calibration_summary = (cal_views.groupby(['analysis', 'protocol', 'model'])[['slope', 'citl', 'ece', 'brier']]
                           .agg(['mean', 'std']).round(4))
display(calibration_summary.loc['main'] if 'main' in calibration_summary.index.get_level_values(0) else calibration_summary)

def ppv_fn(se, sp_, prev):
    with np.errstate(all='ignore'):
        return se * prev / (se * prev + (1 - sp_) * (1 - prev))

def npv_fn(se, sp_, prev):
    with np.errstate(all='ignore'):
        return sp_ * (1 - prev) / (sp_ * (1 - prev) + (1 - se) * prev)

def operating_point(analysis, protocol, model, years=None):
    units = [boot_unit(analysis, model, v['split']) for v in views_of(analysis, protocol)
             if years is None or v['test_year'] in years]
    units = [u for u in units if u is not None]
    if not units:
        return None
    return dict(sens=float(np.mean([u['sens'] for u in units])), spec=float(np.mean([u['spec'] for u in units])),
                sens_b=np.mean([u['sens_b'] for u in units], axis=0), spec_b=np.mean([u['spec_b'] for u in units], axis=0))

def ppv_table(analysis, protocol=PROTO_FWD):
    rows = []
    cy = common_years(analysis, protocol)
    for m in models_in(analysis):
        if m == 'Dummy':
            continue
        op = operating_point(analysis, protocol, m, years=cy)
        if op is None:
            continue
        for prev in PREVALENCES:
            pv, pv_b = ppv_fn(op['sens'], op['spec'], prev), ppv_fn(op['sens_b'], op['spec_b'], prev)
            nv, nv_b = npv_fn(op['sens'], op['spec'], prev), npv_fn(op['sens_b'], op['spec_b'], prev)
            lo, hi = ci(pv_b); nlo, nhi = ci(nv_b)
            alerts = 1000 * (op['sens'] * prev + (1 - op['spec']) * (1 - prev))
            rows.append(dict(Analysis=analysis, Protocol=protocol, Years=_yrs(cy), Model=m, Prevalence=prev, Sensitivity=op['sens'],
                             Specificity=op['spec'], PPV=pv, PPV_lo=lo, PPV_hi=hi, NPV=nv, NPV_lo=nlo, NPV_hi=nhi,
                             Alerts_per_1000=alerts, True_alerts_per_1000=1000 * op['sens'] * prev,
                             False_alerts_per_true_alert=(1 - pv) / pv if pv > 0 else np.nan,
                             **{'PPV (95% CI)': fmt_ci(pv, lo, hi)}))
    return pd.DataFrame(rows)

PPV_TABLE = pd.concat([ppv_table(a, pr) for a in ('main', 'ex2018') if a in ANALYSES
                       for pr in (PROTO_FWD, PROTO_LOYO, PROTO_PD)], ignore_index=True)
_pm = PPV_TABLE[(PPV_TABLE['Analysis'] == 'main') & (PPV_TABLE['Protocol'] == PROTO_FWD)]
display(_pm[_pm['Prevalence'].isin([0.05, 0.10])][['Model', 'Prevalence', 'Sensitivity', 'Specificity', 'PPV (95% CI)',
                                                   'False_alerts_per_true_alert']])
for prev in (0.05, 0.10):
    _r = _pm[_pm['Prevalence'] == prev].sort_values('PPV', ascending=False)
    if len(_r):
        KEY_NUMBERS[f'forward_best_ppv_at_{int(prev*100)}pct'] = dict(
            model=_r.iloc[0]['Model'], ppv=round(_r.iloc[0]['PPV'], 4), ci=[round(_r.iloc[0]['PPV_lo'], 4), round(_r.iloc[0]['PPV_hi'], 4)],
            false_alerts_per_true_alert=round(_r.iloc[0]['False_alerts_per_true_alert'], 2))

### Negative control: user-level label permutation (tuned XGBoost)

In [ ]:
def permute_labels_user_level(pids, labels_, rng):
    out = labels_.copy()
    order = {}
    for i, p in enumerate(pids):
        order.setdefault(p, []).append(i)
    by_len = {}
    for p, ix in order.items():
        by_len.setdefault(len(ix), []).append(p)
    for L, grp in by_len.items():
        if len(grp) == 1:
            ix = np.array(order[grp[0]]); out[ix] = rng.permutation(labels_[ix]); continue
        perm = rng.permutation(len(grp))
        for si, di in enumerate(perm):
            out[np.array(order[grp[di]])] = labels_[np.array(order[grp[si]])]
    return out

perm_results = pd.DataFrame()
if RUN_PERMUTATION_TEST:
    with stage('Permutation negative control'):
        _perm_targets = list(dict.fromkeys(v['split'] for v in views_of('main')
                                           if v['view'] in (f'Forward \u2192 {YEARS[-1]}', 'Participant-disjoint 80/20')))
        def _perm():
            rows = []
            for sp_ in _perm_targets:
                P = get_prep(sp_, SEEDS[0], 'main')
                params = tune('main', 'XGBoost', sp_)['best_params']
                tr_all = np.r_[P['tr'], P['va']]
                y_all = ml_table['label'].to_numpy()[tr_all].astype(int)
                pid_all = ml_table['pid'].to_numpy()[tr_all]
                for k in range(N_PERMUTATIONS):
                    yp = permute_labels_user_level(pid_all, y_all, np.random.default_rng(5000 + k))
                    m_ = make_xgb(params, 5000 + k, yp[:len(P['tr'])]).fit(P['Xtr'], yp[:len(P['tr'])])
                    rows.append(dict(split=sp_, permutation=k, roc_auc=safe_auc(P['yte'], m_.predict_proba(P['Xte'])[:, 1])))
            return pd.DataFrame(rows)
        _perm_fp = fp('perm', PREP_FP, [split_fp(x) for x in _perm_targets],
                      [tune('main', 'XGBoost', x)['fp'] for x in _perm_targets], _perm, permute_labels_user_level,
                      make_xgb, xgb_device_kwargs, N_PERMUTATIONS, SEEDS[0])
        perm_results = cached(f'permutation_test__{_perm_fp}', _perm)
    _obs = RESULTS[(RESULTS['analysis'] == 'main') & (RESULTS['model'] == 'XGBoost')].groupby('split')['roc_auc'].mean()
    perm_summary = (perm_results.groupby('split')['roc_auc'].agg(['mean', 'std', 'max', 'count']).reset_index()
                    .assign(observed=lambda d: d['split'].map(_obs)))
    perm_summary['empirical_p'] = [(1 + (perm_results.loc[perm_results['split'] == s, 'roc_auc'] >= o).sum()) / (1 + c)
                                   for s, o, c in zip(perm_summary['split'], perm_summary['observed'], perm_summary['count'])]
    display(perm_summary)

## 12. Collection-year shift
Year classifiers (participant-grouped, stratified K-fold, out-of-fold AUC) on feature subsets:
all features; without missingness indicators (`_missing` aggregates and history-availability flags) using
median imputation and, more strictly, missingness-blind hot-deck imputation; missingness indicators only;
each modality alone; all but Wi-Fi; and an availability/platform-only set (per-modality observed fraction,
observed days, iOS/Android). For each subset the depression model (tuned XGBoost, simplified pipeline
without the correlation reducer) is also evaluated under forward-chaining, giving the year-vs-depression
contrast of Figure 2. Era analyses contrast pre-COVID (2018-2019) and COVID (2020-2021) cohorts.
Within every fold the channel filter, imputation and classifier are fitted on the training fold only, and the
SHAP decomposition of the year classifier is computed out-of-fold on held-out participants.

In [ ]:
IS_MISS = lambda c: c.endswith('_missing') or c.startswith('hist_avail:')

def add_availability_features(df):
    # Per-window availability summary: observed fraction per modality, observed days and device platform.
    miss_agg = [c for c in AGG_COLS if c.endswith('_missing')]
    new = {}
    for mod in MODALITIES:
        cols = [c for c in miss_agg if modality_of(c) == mod]
        if cols:
            new[f'avail:obs_frac_{mod}'] = (1.0 - df[cols].mean(axis=1)).astype(np.float32)
    new['avail:valid_days'] = df['valid_days'].astype(np.float32)
    new['avail:missing_fraction'] = df['missing_fraction'].astype(np.float32)
    for pf in ('ios', 'android', 'both', 'unknown'):
        new[f'avail:platform_{pf}'] = (df['platform'] == pf).astype(np.float32)
    return new

for _k, _v in add_availability_features(ml_table).items():
    ml_table[_k] = _v
AVAIL_SUMMARY_COLS = [c for c in ml_table.columns if c.startswith('avail:')]

VARIANTS = OrderedDict()
VARIANTS['All features'] = (MAIN_FEATURES, 'median')
VARIANTS['No missingness indicators (median)'] = ([c for c in MAIN_FEATURES if not IS_MISS(c)], 'median')
VARIANTS['No missingness indicators (hot-deck)'] = ([c for c in MAIN_FEATURES if not IS_MISS(c)], 'hotdeck')
VARIANTS['Missingness indicators only'] = ([c for c in MAIN_FEATURES if IS_MISS(c)], 'median')
for _mod in MODALITIES:
    _c = [c for c in MAIN_FEATURES if FEATURE_INFO[c]['modality'] == _mod]
    if _c:
        VARIANTS[f'{_mod} only'] = (_c, 'median')
VARIANTS['All except Wi-Fi'] = ([c for c in MAIN_FEATURES if FEATURE_INFO[c]['modality'] != 'Wi-Fi'], 'median')
VARIANTS['Availability & platform only'] = (AVAIL_SUMMARY_COLS, 'median')
CORE_VARIANTS = ['All features', 'No missingness indicators (median)', 'No missingness indicators (hot-deck)',
                 'Missingness indicators only', 'Availability & platform only']
for _name, (_cols, _imp) in VARIANTS.items():
    FEATURE_SETS[f'var:{_name}'] = _cols
VIEW_ALIAS['domainvar'] = 'main'
VARIANT_SIG = fp({k: list(v) for k, v in VARIANTS.items()})

def year_cv(fs, rows, target, imputation, seed=0, shap_rows=0):
    # Participant-grouped stratified K-fold year classifier. In each fold the channel filter, imputation and model are
    # fitted on the training fold only. With shap_rows > 0, TreeSHAP is computed OUT-OF-FOLD on a sample of the
    # held-out windows and accumulated per feature.
    X = feature_matrix('ml', fs)
    groups = ml_table['pid'].to_numpy()[rows]
    classes = np.unique(target)
    yk = np.searchsorted(classes, target)
    oof = np.zeros((len(rows), len(classes)))
    acc = np.zeros(len(FEATURE_SETS[fs]))
    rng = np.random.default_rng(seed)
    skf = StratifiedGroupKFold(n_splits=DOMAIN_CV_FOLDS, shuffle=True, random_state=seed)
    for k, (a, b) in enumerate(skf.split(rows, yk, groups)):
        cols = allowed_columns(fs, channel_keep_mask('ml', rows[a]))
        Xa, Xb = X[np.ix_(rows[a], cols)], X[np.ix_(rows[b], cols)]
        if imputation == 'hotdeck':
            Xa, Xb = hotdeck_impute(Xa, [Xa, Xb], np.random.default_rng(seed + k))
        else:
            imp = MedianImputer().fit(Xa)
            Xa, Xb = imp.transform(Xa), imp.transform(Xb)
        clf = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8,
                                random_state=seed, n_jobs=-1, verbosity=0, **xgb_device_kwargs())
        clf.fit(Xa, yk[a])
        oof[b] = clf.predict_proba(Xb)
        if shap_rows:
            booster = clf.get_booster()
            sel = rng.choice(len(b), size=min(len(b), int(np.ceil(shap_rows / DOMAIN_CV_FOLDS))), replace=False)
            for i in range(0, len(sel), 250):
                c = np.abs(booster.predict(xgb.DMatrix(Xb[sel[i:i + 250]]), pred_contribs=True)[..., :-1])
                acc[cols] += c.reshape(-1, c.shape[-1]).sum(axis=0)
    out = dict(shap=acc if shap_rows else None)
    if len(classes) == 2:
        out.update(auc=float(roc_auc_score(yk, oof[:, 1])), per_class={})
    else:
        out.update(auc=float(roc_auc_score(yk, oof, multi_class='ovr', average='macro')),
                   per_class={int(c): float(roc_auc_score(yk == i, oof[:, i])) for i, c in enumerate(classes)})
    return out

def _domain_year_analysis():
    yrs = ml_table['year'].to_numpy()
    all_rows = np.arange(len(ml_table))
    rows, shap_vec = [], None
    for name, (cols, imp) in VARIANTS.items():
        if not cols:
            continue
        fs = f'var:{name}'
        r = year_cv(fs, all_rows, yrs, imp, shap_rows=YEAR_SHAP_ROWS if name == 'All features' else 0)
        if name == 'All features':
            shap_vec = r['shap']
        rec = dict(Variant=name, Features=len(cols), Imputation=imp, **{'Year AUC (macro OvR)': r['auc']},
                   **{f'OvR {y}': v for y, v in r['per_class'].items()})
        if name in CORE_VARIANTS:
            pre, cov = [y for y in YEARS if y <= 2019], [y for y in YEARS if y >= 2020]
            if pre and cov:
                rec['Pre-COVID vs COVID AUC'] = year_cv(fs, all_rows, (yrs >= 2020).astype(int), imp)['auc']
            for a_, b_ in [(2018, 2019), (2020, 2021)]:
                m = np.isin(yrs, [a_, b_])
                if m.sum() and len(np.unique(yrs[m])) == 2:
                    rec[f'{a_} vs {b_} AUC'] = year_cv(fs, all_rows[m], yrs[m], imp)['auc']
            m = yrs != 2018
            if (yrs == 2018).any() and len(np.unique(yrs[m])) > 2:
                rec['Year AUC excl. 2018 (macro OvR)'] = year_cv(fs, all_rows[m], yrs[m], imp)['auc']
        rows.append(rec)
        print(f"  {name:42s} year AUC {rec['Year AUC (macro OvR)']:.3f}")
    df = pd.DataFrame({'feature': FEATURE_SETS['var:All features'], 'abs_shap': shap_vec / max(shap_vec.sum(), 1e-12)})
    df['Modality'] = [FEATURE_INFO[f]['modality'] for f in df['feature']]
    df['Feature type'] = [FEATURE_INFO[f]['ftype'] for f in df['feature']]
    shares = df.groupby(['Modality', 'Feature type'])['abs_shap'].sum().unstack(fill_value=0.0)
    return pd.DataFrame(rows), shares

def _domain_dep_preds():
    out = {}
    fwd = [v for v in views_of('main', PROTO_FWD)]
    for name, (cols, imp) in VARIANTS.items():
        if not cols:
            continue
        for v in fwd:
            params = tune('main', 'XGBoost', v['split'])['best_params']
            for seed in SEEDS[:DOMAIN_DEP_SEEDS]:
                P = get_prep(v['split'], seed, f'var:{name}', imputation=imp, reduce=False)
                o = fit_xgb(P, params, seed)
                o.update(yva=P['yva'], yte=P['yte'], pids_te=P['pids_te'], years_te=P['years_te'])
                _, pdf = _record('domainvar', name, v['split'], seed, o, params)
                out[('domainvar', name, v['split'], seed)] = pdf
    return out


ERA_DEFS = [('Within pre-COVID', [2018], 2019), ('Pre-COVID \u2192 COVID', [2018, 2019], 2020),
            ('Pre-COVID \u2192 COVID', [2018, 2019], 2021), ('Within COVID', [2020], 2021),
            ('COVID \u2192 pre-COVID', [2020, 2021], 2018), ('COVID \u2192 pre-COVID', [2020, 2021], 2019)]

def register_era_splits():
    if any(v['analysis'] == 'era' for v in VIEWS):          # idempotent if the cell is re-run
        return [v['split'] for v in VIEWS if v['analysis'] == 'era']
    yrs = ml_table['year'].to_numpy()
    names = []
    for cat, tr_y, te_y in ERA_DEFS:
        if (set(tr_y) | {te_y}) <= set(YEARS):
            names.append(register_split('era', 'ml', PROTO_ERA, f'{cat}: {"+".join(map(str, tr_y))} \u2192 {te_y}', te_y,
                                        f'era|{"_".join(map(str, tr_y))}_to_{te_y}',
                                        np.flatnonzero(np.isin(yrs, tr_y)), np.flatnonzero(yrs == te_y), None))
    return names

ERA_LAST_FWD = [v['split'] for v in views_of('main', PROTO_FWD)][-1]

def _era_preds():
    out = {}
    params = tune('main', 'XGBoost', ERA_LAST_FWD)['best_params']
    for name in ERA_SPLITS:
        for seed in SEEDS[:DOMAIN_DEP_SEEDS]:
            P = get_prep(name, seed, 'main', reduce=False)
            o = fit_xgb(P, params, seed)
            o.update(yva=P['yva'], yte=P['yte'], pids_te=P['pids_te'], years_te=P['years_te'])
            out[('era', 'XGBoost', name, seed)] = _record('era', 'XGBoost', name, seed, o, params)[1]
    return out

domain_year_table = pd.DataFrame(); year_shap_shares = pd.DataFrame(); era_table = pd.DataFrame()
domain_variant_table = pd.DataFrame()
if RUN_DOMAIN_SHIFT:
    _fwd_dom = [v['split'] for v in views_of('main', PROTO_FWD)]
    with stage('Domain shift: year classifiers (out-of-fold AUC and SHAP)'):
        _dfp = fp('domain_year', DATA_FP, VARIANT_SIG, add_availability_features, year_cv, _domain_year_analysis,
                  MedianImputer, hotdeck_impute, channel_keep_mask, allowed_columns, xgb_device_kwargs,
                  DOMAIN_CV_FOLDS, YEAR_SHAP_ROWS, MAX_MISSING_FRAC)
        domain_year_table, year_shap_shares = cached(f'domain_year__{_dfp}', _domain_year_analysis)
    with stage('Domain shift: depression AUC on the same feature subsets'):
        _dfp = fp('domainvar', PREP_FP, VARIANT_SIG, add_availability_features, _domain_dep_preds, fit_xgb, make_xgb,
                  _record, [split_fp(x) for x in _fwd_dom], [tune('main', 'XGBoost', x)['fp'] for x in _fwd_dom],
                  SEEDS[:DOMAIN_DEP_SEEDS])
        PRED.update(cached(f'domainvar_preds__{_dfp}', _domain_dep_preds))
    with stage('Domain shift: era transfer'):
        ERA_SPLITS = register_era_splits()
        _dfp = fp('era', PREP_FP, ERA_DEFS, register_era_splits, _era_preds, fit_xgb, make_xgb, _record,
                  [split_fp(x) for x in ERA_SPLITS], tune('main', 'XGBoost', ERA_LAST_FWD)['fp'], SEEDS[:DOMAIN_DEP_SEEDS])
        PRED.update(cached(f'era_preds__{_dfp}', _era_preds))
    _rows = []
    for name in VARIANTS:
        r = pooled('domainvar', PROTO_FWD, name)
        if r is None:
            continue
        yr = domain_year_table.set_index('Variant').loc[name] if name in set(domain_year_table['Variant']) else None
        _rows.append({'Variant': name, 'Features': len(VARIANTS[name][0]),
                      'Year AUC (macro OvR)': yr['Year AUC (macro OvR)'] if yr is not None else np.nan,
                      'Depression AUC (forward, pooled)': r['point'], 'Dep AUC lo': r['lo'], 'Dep AUC hi': r['hi']})
    domain_variant_table = pd.DataFrame(_rows)
    _rows = []
    for v in [v for v in VIEWS if v['analysis'] == 'era']:
        b = boot_unit('era', 'XGBoost', v['split'])
        if b is not None:
            lo, hi = ci(b['auc_b'])
            _rows.append({'Era contrast': v['view'].split(':')[0], 'Train \u2192 test': v['view'].split(': ')[1],
                          'AUC': b['auc'], 'AUC_lo': lo, 'AUC_hi': hi})
    era_table = pd.DataFrame(_rows)
    display(domain_year_table.round(3)); display(domain_variant_table.round(3)); display(era_table.round(3))
    display((100 * year_shap_shares).round(1))
    if len(domain_year_table):
        _d = domain_year_table.set_index('Variant')
        KEY_NUMBERS['year_auc_all_features'] = round(float(_d.loc['All features', 'Year AUC (macro OvR)']), 4)
        for _k in ['No missingness indicators (median)', 'No missingness indicators (hot-deck)', 'Availability & platform only',
                   'All except Wi-Fi', 'Wi-Fi only']:
            if _k in _d.index:
                KEY_NUMBERS[f'year_auc::{_k}'] = round(float(_d.loc[_k, 'Year AUC (macro OvR)']), 4)
        for _k in ['Pre-COVID vs COVID AUC', '2018 vs 2019 AUC', '2020 vs 2021 AUC', 'Year AUC excl. 2018 (macro OvR)']:
            if _k in _d.columns:
                KEY_NUMBERS[f'year_auc::{_k}'] = round(float(_d.loc['All features', _k]), 4)
    if len(year_shap_shares):
        KEY_NUMBERS['year_classifier_shap_share_missingness'] = round(float(year_shap_shares.get('missingness', pd.Series(0)).sum()), 4)
        KEY_NUMBERS['year_classifier_shap_share_wifi'] = round(float(year_shap_shares.loc['Wi-Fi'].sum()) if 'Wi-Fi' in year_shap_shares.index else 0.0, 4)

## 13. Explanation stability
Four comparisons of XGBoost TreeSHAP rankings, each computed for every seed:
1. **Within-year split-half**: same model, same year, random halves of test participants (sampling noise).
2. **Seed refit**: same year, models refitted with different seeds (training noise).
3. **Same model, different years**: the participant-disjoint model explained separately on each year's
   test participants (distribution effect with the model held fixed), with a permutation test that
   shuffles participants between years while preserving group sizes.
4. **Cross-year**: year-specific LOYO models, each explained on its own held-out year.

Metrics: Jaccard@20 of top items; top-100 Spearman (rank agreement within each ranking's own top-100, averaged
over both directions: ~0 for unrelated rankings and not inflated by tied zero-importance features); full-list
Spearman (reported for completeness, but inflated by ties); total-variation distance between importance shares.
Levels: pooled correlation cluster, behavioural construct, modality.

In [ ]:
def imp_vec(entry, level, mask=None):
    sums, rows = entry['levels'][level], entry['pid_rows']
    v = (sums[mask].sum(0) / max(rows[mask].sum(), 1)) if mask is not None else sums.sum(0) / max(rows.sum(), 1)
    v = v.astype(np.float64)
    v[~entry['present'][level]] = np.nan
    return v

def compare_vecs(a, b, level):
    pa, pb = np.isfinite(a), np.isfinite(b)
    out = dict(jaccard=np.nan, spearman=np.nan, spearman_top=np.nan, tv_distance=np.nan)
    common = pa & pb
    if level in ('cluster', 'construct'):
        ia, ib = np.flatnonzero(pa), np.flatnonzero(pb)
        ta, tb = set(ia[np.argsort(-a[ia], kind='stable')[:TOP_K]]), set(ib[np.argsort(-b[ib], kind='stable')[:TOP_K]])
        out['jaccard'] = len(ta & tb) / max(len(ta | tb), 1)
    if common.sum() > 2:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            out['spearman'] = spearmanr(a[common], b[common])[0]
            ic = np.flatnonzero(common)
            # Symmetric conditional top-k Spearman: rank agreement within A's top-k and within B's top-k, averaged.
            # (Spearman over the UNION of both top-k sets is biased negative under independence: ~ -0.7.)
            if level in ('cluster', 'construct'):
                ia_, ib_ = ic[np.argsort(-a[ic], kind='stable')[:TOP_UNION]], ic[np.argsort(-b[ic], kind='stable')[:TOP_UNION]]
                out['spearman_top'] = float(np.nanmean([spearmanr(a[ia_], b[ia_])[0], spearmanr(a[ib_], b[ib_])[0]]))
            else:
                out['spearman_top'] = out['spearman']
        sa, sb = a[common] / max(a[common].sum(), 1e-12), b[common] / max(b[common].sum(), 1e-12)
        out['tv_distance'] = 0.5 * float(np.abs(sa - sb).sum())
    return out

def kendall_w(M):
    M = M[np.isfinite(M).all(1)]
    n, k = M.shape
    if n < 3 or k < 2:
        return np.nan
    R = np.column_stack([pd.Series(-M[:, j]).rank().to_numpy() for j in range(k)])
    S = ((R.sum(1) - k * (n + 1) / 2) ** 2).sum()
    return float(12 * S / (k ** 2 * (n ** 3 - n)))

STAB_LEVELS = ['cluster', 'construct', 'modality']
CMP_LABELS = {'A': 'Within-year split-half', 'B': 'Seed refit (same year)', 'C': 'Same model, different years',
              'D': 'Cross-year (year-specific models)'}

def stability_pairs(an):
    rows, extra = [], []
    loyo = {v['test_year']: v['split'] for v in views_of(an, PROTO_LOYO)}
    pd_split = [v['split'] for v in views_of(an, PROTO_PD)]
    ent = lambda s, seed: SHAP_STORE.get((an, s, seed))
    for y, s in loyo.items():
        for seed in SEEDS:
            e = ent(s, seed)
            if e is None:
                continue
            rng = np.random.default_rng(stable_seed('half', an, y, seed))
            for r in range(SPLIT_HALF_REPEATS):
                half = rng.permutation(len(e['pids']))[:len(e['pids']) // 2]
                mask = np.zeros(len(e['pids']), bool); mask[half] = True
                for lvl in STAB_LEVELS:
                    rows.append(dict(analysis=an, comparison='A', level=lvl, years=f'{y}', seed=seed,
                                     **compare_vecs(imp_vec(e, lvl, mask), imp_vec(e, lvl, ~mask), lvl)))
        for s1, s2 in combinations(SEEDS, 2):
            e1, e2 = ent(s, s1), ent(s, s2)
            if e1 is None or e2 is None:
                continue
            for lvl in STAB_LEVELS:
                rows.append(dict(analysis=an, comparison='B', level=lvl, years=f'{y}', seed=f'{s1}-{s2}',
                                 **compare_vecs(imp_vec(e1, lvl), imp_vec(e2, lvl), lvl)))
    for y1, y2 in combinations(sorted(loyo), 2):
        for seed in SEEDS:
            e1, e2 = ent(loyo[y1], seed), ent(loyo[y2], seed)
            if e1 is None or e2 is None:
                continue
            for lvl in STAB_LEVELS:
                rows.append(dict(analysis=an, comparison='D', level=lvl, years=f'{y1}-{y2}', seed=seed,
                                 **compare_vecs(imp_vec(e1, lvl), imp_vec(e2, lvl), lvl)))
    if pd_split:
        for seed in SEEDS:
            e = ent(pd_split[0], seed)
            if e is None:
                continue
            ys = sorted(np.unique(e['pid_year']))
            for lvl in STAB_LEVELS:
                obs = []
                for y1, y2 in combinations(ys, 2):
                    r_ = compare_vecs(imp_vec(e, lvl, e['pid_year'] == y1), imp_vec(e, lvl, e['pid_year'] == y2), lvl)
                    rows.append(dict(analysis=an, comparison='C', level=lvl, years=f'{y1}-{y2}', seed=seed, **r_))
                    obs.append(r_)
                metric = 'jaccard' if lvl != 'modality' else 'spearman'
                o = np.nanmean([r_[metric] for r_ in obs]) if obs else np.nan
                rng = np.random.default_rng(stable_seed('perm', an, seed, lvl))
                null = []
                for _ in range(YEAR_PARTITION_PERMUTATIONS):
                    py = rng.permutation(e['pid_year'])
                    null.append(np.nanmean([compare_vecs(imp_vec(e, lvl, py == y1), imp_vec(e, lvl, py == y2), lvl)[metric]
                                            for y1, y2 in combinations(ys, 2)]))
                null = np.asarray(null)
                extra.append(dict(analysis=an, level=lvl, seed=seed, metric=metric, observed=o, null_mean=float(np.nanmean(null)),
                                  p_lower=float((1 + np.sum(null <= o)) / (1 + len(null)))))
    return pd.DataFrame(rows), pd.DataFrame(extra)

def stability_summary(df):
    g = df.groupby(['analysis', 'comparison', 'level'])
    out = []
    for (an, c, lvl), d in g:
        rec = dict(analysis=an, comparison=CMP_LABELS[c], code=c, level=lvl, n=len(d))
        for m in ('jaccard', 'spearman_top', 'spearman', 'tv_distance'):
            v = d[m].dropna()
            rec[m] = v.mean() if len(v) else np.nan
            rec[f'{m}_lo'], rec[f'{m}_hi'] = (np.percentile(v, 2.5), np.percentile(v, 97.5)) if len(v) else (np.nan, np.nan)
        out.append(rec)
    return pd.DataFrame(out)

STAB_PAIRS, STAB_PERM, STAB_SUMMARY = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
kendall_table, modality_shares, top_clusters, top_constructs = (pd.DataFrame() for _ in range(4))
if RUN_EXPLANATION_STABILITY and SHAP_STORE:
    with stage('Explanation stability'):
        def _stab():
            ps, ex = zip(*[stability_pairs(a) for a in ('main', 'ex2018') if a in ANALYSES])
            return pd.concat(ps, ignore_index=True), pd.concat(ex, ignore_index=True)
        _sfp = fp('stability', sorted((f'{a}|{x}', FINAL_FP[(a, m_, x)]) for (a, m_, x) in FINAL_FP
                                      if m_ == 'XGBoost' and a in ('main', 'ex2018')),
                  _stab, stability_pairs, imp_vec, compare_vecs, SPLIT_HALF_REPEATS, YEAR_PARTITION_PERMUTATIONS,
                  TOP_K, TOP_UNION, SEEDS, STAB_LEVELS)
        STAB_PAIRS, STAB_PERM = cached(f'stability_pairs__{_sfp}', _stab)
        STAB_SUMMARY = stability_summary(STAB_PAIRS)
        _kw = []
        for an in [a for a in ('main', 'ex2018') if a in ANALYSES]:
            loyo = {v['test_year']: v['split'] for v in views_of(an, PROTO_LOYO)}
            for lvl in STAB_LEVELS:
                for seed in SEEDS:
                    es = [SHAP_STORE.get((an, s, seed)) for s in loyo.values()]
                    if all(e is not None for e in es):
                        _kw.append(dict(analysis=an, level=lvl, reference='Cross-year (4 LOYO models)' if an == 'main' else 'Cross-year (3 LOYO models)',
                                        W=kendall_w(np.column_stack([imp_vec(e, lvl) for e in es]))))
                for y, s in loyo.items():
                    es = [SHAP_STORE.get((an, s, seed)) for seed in SEEDS]
                    if all(e is not None for e in es) and len(es) > 1:
                        _kw.append(dict(analysis=an, level=lvl, reference=f'Across seeds, year {y}',
                                        W=kendall_w(np.column_stack([imp_vec(e, lvl) for e in es]))))
        kendall_table = (pd.DataFrame(_kw).groupby(['analysis', 'level', 'reference'])['W'].agg(['mean', 'std', 'count'])
                         .reset_index())
        _ms = []
        for v in views_of('main', PROTO_LOYO):
            for seed in SEEDS:
                e = SHAP_STORE.get(('main', v['split'], seed))
                if e is None:
                    continue
                iv = imp_vec(e, 'modality')
                iv = np.nan_to_num(iv) / max(np.nansum(iv), 1e-12)
                for mod, val in zip(ITEMS['modality'], iv):
                    _ms.append(dict(year=v['test_year'], seed=seed, modality=mod, share=val))
        modality_shares = pd.DataFrame(_ms)
        _last = [v['split'] for v in views_of('main', PROTO_FWD)][-1]
        for lvl, name in (('cluster', 'top_clusters'), ('construct', 'top_constructs')):
            vecs, signed = [], []
            for seed in SEEDS:
                e = SHAP_STORE.get(('main', _last, seed))
                if e is not None:
                    vecs.append(imp_vec(e, lvl)); signed.append(e['signed'][lvl])
            if vecs:
                V = np.vstack(vecs)
                ranks = np.vstack([pd.Series(-r).rank(method='min').to_numpy() for r in V])
                t = pd.DataFrame({'item': ITEMS[lvl], 'mean_abs_shap': np.nanmean(V, 0), 'mean_signed_shap': np.mean(signed, 0),
                                  'median_rank': np.nanmedian(ranks, 0), 'top20_frequency': (ranks <= TOP_K).mean(0)})
                t['modality'] = [modality_of(i.split(':', 1)[0]) if ':' in i else i for i in t['item']]
                t = t.dropna(subset=['mean_abs_shap']).sort_values('mean_abs_shap', ascending=False).head(40)
                if lvl == 'cluster':
                    top_clusters = t
                else:
                    top_constructs = t
    display(STAB_SUMMARY[STAB_SUMMARY['analysis'] == 'main'].round(3))
    display(STAB_PERM.groupby(['analysis', 'level'])[['observed', 'null_mean', 'p_lower']].mean().round(3))
    display(kendall_table.round(3))
    _s = STAB_SUMMARY[(STAB_SUMMARY['analysis'] == 'main') & (STAB_SUMMARY['level'] == 'cluster')].set_index('code')
    for c in _s.index:
        KEY_NUMBERS[f'stability_cluster_{CMP_LABELS[c]}'] = dict(
            jaccard20=[round(_s.loc[c, k], 4) for k in ('jaccard', 'jaccard_lo', 'jaccard_hi')],
            spearman_top=[round(_s.loc[c, k], 4) for k in ('spearman_top', 'spearman_top_lo', 'spearman_top_hi')],
            spearman_full=round(_s.loc[c, 'spearman'], 4))
    _s = STAB_SUMMARY[(STAB_SUMMARY['analysis'] == 'main')].set_index(['code', 'level'])
    for lvl in STAB_LEVELS:
        if ('D', lvl) in _s.index and ('A', lvl) in _s.index:
            KEY_NUMBERS[f'spearman_top_{lvl}_crossyear_vs_splithalf'] = [round(_s.loc[('D', lvl), 'spearman_top'], 4),
                                                                         round(_s.loc[('A', lvl), 'spearman_top'], 4)]

## 14. Sensitivity analyses: 2018 labels, consistent BDI-II outcome, `_norm` ablation

In [ ]:
def _row(label, main_val, ex_val, note=''):
    return {'Quantity': label, 'Main analysis (2018 included)': main_val, 'Sensitivity (2018 excluded)': ex_val, 'Note': note}

sens_rows, sens_model_rows = [], []
if 'ex2018' in ANALYSES:
    common_fwd = [v['test_year'] for v in views_of('ex2018', PROTO_FWD)]
    common_loyo = [v['test_year'] for v in views_of('ex2018', PROTO_LOYO)]
    for proto, yrs_ in ((PROTO_FWD, common_fwd), (PROTO_LOYO, common_loyo)):
        for m in models_in('ex2018'):
            ym = sorted(set(model_years('main', proto, m)) & set(model_years('ex2018', proto, m)) & set(yrs_))
            if not ym:
                continue
            a, b = pooled('main', proto, m, years=ym), pooled('ex2018', proto, m, years=ym)
            d = paired('ex2018', m, 'main', m, proto, years=ym)
            if a is None or b is None:
                continue
            sens_model_rows.append({'Protocol': proto, 'Test years': ', '.join(map(str, ym)), 'Model': m,
                                    'Main AUC (95% CI)': fmt_ci(a['point'], a['lo'], a['hi']),
                                    'Excl. 2018 AUC (95% CI)': fmt_ci(b['point'], b['lo'], b['hi']),
                                    'Delta (excl - main)': d['delta'] if d else np.nan,
                                    'Delta CI': f"[{d['lo']:.3f}, {d['hi']:.3f}]" if d else '', 'p': d['p'] if d else np.nan})
    sens_model_table = pd.DataFrame(sens_model_rows)
    if len(sens_model_table):
        sens_model_table['p_Holm'] = holm(sens_model_table['p'].to_numpy())
    def _best(an, proto, yrs_):
        r = [(m, pooled(an, proto, m, years=yrs_)) for m in models_in(an) if m != 'Dummy']
        r = [(m, x) for m, x in r if x is not None and len(x['years']) == len(yrs_)]
        return max(r, key=lambda t: t[1]['point']) if r else (None, None)
    for proto, yrs_ in ((PROTO_FWD, common_fwd), (PROTO_LOYO, common_loyo)):
        bm, bx = _best('main', proto, yrs_); be, bxe = _best('ex2018', proto, yrs_)
        if bm:
            sens_rows.append(_row(f'Best {proto} AUC ({", ".join(map(str, yrs_))})', f"{bm}: {fmt_ci(bx['point'], bx['lo'], bx['hi'])}",
                                  f"{be}: {fmt_ci(bxe['point'], bxe['lo'], bxe['hi'])}"))
    cm, ce = COMPARISONS.get('main', pd.DataFrame()), COMPARISONS.get('ex2018', pd.DataFrame())
    for fam in ('vs XGBoost', 'DG vs ERM'):
        f = lambda c: c[(c['Protocol'] == PROTO_FWD) & (c['Family'] == fam)] if len(c) else c
        sens_rows.append(_row(f'Forward: comparisons {fam} significant after Holm',
                              f"{int(f(cm)['Significant (Holm)'].sum())}/{len(f(cm))}" if len(f(cm)) else '',
                              f"{int(f(ce)['Significant (Holm)'].sum())}/{len(f(ce))}" if len(f(ce)) else '',
                              note=f"test years: main {_yrs(common_years('main', PROTO_FWD))}; "
                                   f"excl. 2018 {_yrs(common_years('ex2018', PROTO_FWD))}"))
    for m in MULTI_DOMAIN_ONLY:
        ym = sorted(set(model_years('main', PROTO_FWD, m)) & set(model_years('ex2018', PROTO_FWD, m)))
        rm = paired('main', m, 'main', 'MLP-ERM', PROTO_FWD, years=ym) if ym else None
        re_ = paired('ex2018', m, 'ex2018', 'MLP-ERM', PROTO_FWD, years=ym) if ym else None
        if rm and re_:
            sens_rows.append(_row(f'Forward {m} - MLP-ERM (test years {", ".join(map(str, ym))})',
                                  f"{rm['delta']:+.3f} [{rm['lo']:+.3f}, {rm['hi']:+.3f}]",
                                  f"{re_['delta']:+.3f} [{re_['lo']:+.3f}, {re_['hi']:+.3f}]"))
    if len(STAB_PAIRS):
        for code in ('A', 'D'):
            for lvl, met in (('cluster', 'jaccard'), ('cluster', 'spearman_top'), ('modality', 'spearman')):
                d0 = STAB_PAIRS[(STAB_PAIRS['comparison'] == code) & (STAB_PAIRS['level'] == lvl)]
                d0 = d0[~d0['years'].str.contains('2018')]
                vm = d0[d0['analysis'] == 'main'][met].mean(); ve = d0[d0['analysis'] == 'ex2018'][met].mean()
                sens_rows.append(_row(f'{CMP_LABELS[code]}: {met} ({lvl}), 2019-2021 only', round(vm, 3), round(ve, 3)))
    if len(domain_year_table) and 'Year AUC excl. 2018 (macro OvR)' in domain_year_table.columns:
        _d = domain_year_table.set_index('Variant')
        sens_rows.append(_row('Year-classifier macro AUC (all features)', round(_d.loc['All features', 'Year AUC (macro OvR)'], 3),
                              round(_d.loc['All features', 'Year AUC excl. 2018 (macro OvR)'], 3)))
sens_summary = pd.DataFrame(sens_rows)
if 'bdi' in ANALYSES:
    for _, r in POOLED_TABLES['bdi'][POOLED_TABLES['bdi']['Protocol'] == PROTO_FWD].iterrows():
        sens_rows.append({'Quantity': f'BDI-II end-term outcome, forward AUC: {r["Model"]}',
                          'Main analysis (2018 included)': r['AUC (95% CI)'], 'Sensitivity (2018 excluded)': '',
                          'Note': 'consistent outcome in all four years'})
    sens_summary = pd.DataFrame(sens_rows)
display(sens_summary)
if 'ex2018' in ANALYSES:
    display(sens_model_table)

norm_ablation_table = pd.DataFrame()
if 'normabl' in ANALYSES:
    _r = []
    for proto in (PROTO_FWD, PROTO_LOYO, PROTO_PD):
        for m in ANALYSES['normabl']['models']:
            a, b = pooled('main', proto, m), pooled('normabl', proto, m)
            d = paired('normabl', m, 'main', m, proto)
            if a and b and d:
                _r.append({'Protocol': proto, 'Model': m, '_norm included': fmt_ci(a['point'], a['lo'], a['hi']),
                           '_norm excluded': fmt_ci(b['point'], b['lo'], b['hi']), 'Delta (excl - incl)': d['delta'],
                           'CI_lo': d['lo'], 'CI_hi': d['hi'], 'p': d['p']})
    norm_ablation_table = pd.DataFrame(_r)
    if len(norm_ablation_table):
        norm_ablation_table['p_Holm'] = holm(norm_ablation_table['p'].to_numpy())
    display(norm_ablation_table)

## 15. Main figures (1-5)

In [ ]:
def short_variant(n):
    return (n.replace('No missingness indicators', 'No missingness').replace('Availability & platform only', 'Availability/platform')
             .replace('Missingness indicators only', 'Missingness only').replace(' only', ''))

def model_order(ms):
    return [m for m in MODELS_ALL if m in ms]

def _yr_range(ys):
    ys = [y for y in ys if y is not None]
    if len(ys) > 1 and ys == list(range(ys[0], ys[-1] + 1)):
        return f'{ys[0]}\u2013{ys[-1]}'
    return _yrs(ys)

# ---- Figure 1: evaluation framework ---------------------------------------
fig, ax = plt.subplots(figsize=(7.4, 4.8))
COL = {'train': '#0072B2', 'test': '#D55E00', 'unused': '#E3E3E3'}
rows = [('Forward-chaining (primary)', f'Forward \u2192 {y}', {z: 'test' if z == y else 'train' if z < y else 'unused' for z in YEARS})
        for y in YEARS[1:]]
rows += [('Leave-one-year-out (domain transfer)', f'Leave-out {y}', {z: 'test' if z == y else 'train' for z in YEARS}) for y in YEARS]
rows += [('Participant-disjoint (reference)', '80/20 participants', {z: 'pd' for z in YEARS})]
nR = len(rows)
for i, (grp, lab, cells) in enumerate(rows):
    yy = nR - 1 - i
    for j, z in enumerate(YEARS):
        if cells[z] == 'pd':
            ax.add_patch(Rectangle((j + 0.05, yy + 0.1), 0.9 * 0.8, 0.8, color=COL['train']))
            ax.add_patch(Rectangle((j + 0.05 + 0.72, yy + 0.1), 0.9 * 0.2, 0.8, color=COL['test']))
        else:
            ax.add_patch(Rectangle((j + 0.05, yy + 0.1), 0.9, 0.8, color=COL[cells[z]]))
        if z == 2018:
            ax.add_patch(Rectangle((j + 0.05, yy + 0.1), 0.9, 0.8, fill=False, hatch='///', ec='white', lw=0))
    ax.text(-0.1, yy + 0.5, lab, ha='right', va='center', fontsize=8)
_grp_pos = {}
for i, (grp, _, _) in enumerate(rows):
    _grp_pos.setdefault(grp, []).append(nR - 1 - i)
for grp, ys in _grp_pos.items():
    ax.plot([-2.25, -2.25], [min(ys) + 0.1, max(ys) + 0.9], color='black', lw=1)
    ax.text(-2.35, np.mean(ys) + 0.5, grp, ha='right', va='center', fontsize=8.5, fontweight='bold')
for j, z in enumerate(YEARS):
    ax.text(j + 0.5, nR + 0.15, str(z), ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xlim(-5.3, len(YEARS) + 0.1); ax.set_ylim(-2.6, nR + 0.7)
ax.axis('off')
ax.legend(handles=[Patch(color=COL['train'], label='Outer training data'), Patch(color=COL['test'], label='Outer test data (scored once per seed)'),
                   Patch(color=COL['unused'], label='Not used'),
                   Patch(facecolor='#777777', hatch='///', edgecolor='white', label='2018: PANAS-derived weekly labels (excluded in sensitivity)')],
          loc='upper center', bbox_to_anchor=(0.5, 0.17), ncol=2, frameon=False, fontsize=7.5)
ax.text((len(YEARS) - 5.3) / 2, -2.35, 'Inside every outer training set: participant-disjoint 80/20 inner split for Optuna tuning, early stopping\n'
        f'and threshold selection. {len(SEEDS)} seeds per model; two-level (participant x seed) bootstrap for uncertainty.',
        ha='center', va='center', fontsize=7.5, style='italic')
save_fig(fig, 'Fig1_evaluation_framework', 'Evaluation framework: forward-chaining (primary), leave-one-year-out and participant-disjoint protocols.', main=True)
plt.show()

# ---- Figure 2: collection-year AUC versus depression-risk AUC --------------
if len(domain_year_table):
    fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.0), gridspec_kw={'width_ratios': [1.0, 1.25]})
    ax = axes[0]
    _all = domain_year_table.set_index('Variant').loc['All features']
    xs, labs = [], []
    for j, y in enumerate(YEARS):
        v = [v for v in views_of('main', PROTO_FWD) if v['test_year'] == y] or [v for v in views_of('main', PROTO_LOYO) if v['test_year'] == y]
        b = boot_unit('main', 'XGBoost', v[0]['split'])
        lo, hi = ci(b['auc_b'])
        ax.bar(j - 0.18, _all.get(f'OvR {y}', np.nan), width=0.34, color='#CC79A7', label='Collection year (one-vs-rest)' if j == 0 else None)
        ax.bar(j + 0.18, b['auc'], width=0.34, color='#0072B2', yerr=[[b['auc'] - lo], [hi - b['auc']]], capsize=3,
               label='Depression risk (XGBoost)' if j == 0 else None)
        labs.append(v[0]['view'].replace('Leave-out', 'LOYO').replace('Forward ', 'Fwd '))
    ax.axhline(0.5, color='black', ls='--', lw=1)
    ax.set_xticks(range(len(YEARS))); ax.set_xticklabels(labs, fontsize=7.5)
    ax.set_ylim(0.4, 1.16); ax.set_yticks(np.arange(0.4, 1.01, 0.1)); ax.set_ylabel('ROC AUC')
    ax.set_title('A  Collection-year vs depression-risk AUC by test year', loc='left')
    ax.legend(frameon=False, loc='upper center', ncol=2, fontsize=7.5)
    ax = axes[1]
    dv = domain_variant_table.dropna(subset=['Year AUC (macro OvR)'])
    for _, r in dv.iterrows():
        c = '#D55E00' if 'issingness' in r['Variant'] or 'Availability' in r['Variant'] else '#0072B2' if r['Variant'] in ('All features', 'All except Wi-Fi') else '#009E73'
        ax.errorbar(r['Year AUC (macro OvR)'], r['Depression AUC (forward, pooled)'],
                    yerr=[[r['Depression AUC (forward, pooled)'] - r['Dep AUC lo']], [r['Dep AUC hi'] - r['Depression AUC (forward, pooled)']]],
                    fmt='o', color=c, ms=5, capsize=2, lw=1)
    ax.plot([0.5, 1.0], [0.5, 1.0], color='grey', ls=':', lw=1)
    ax.axhline(0.5, color='black', ls='--', lw=1)
    ax.set_xlim(0.45, 1.02); ax.set_ylim(0.45, max(0.75, dv['Dep AUC hi'].max() + 0.03) if len(dv) else 0.75)
    ax.set_xlabel('Collection-year classifier AUC (macro one-vs-rest)')
    ax.set_ylabel('Depression-risk AUC (forward-chaining, pooled)')
    ax.set_title('B  Year vs depression AUC across feature subsets', loc='left')
    _ylo, _yhi = ax.get_ylim()
    _dvs = dv.sort_values('Depression AUC (forward, pooled)')
    for (_, r), yl in zip(_dvs.iterrows(), np.linspace(_ylo + 0.03 * (_yhi - _ylo), _yhi - 0.30 * (_yhi - _ylo), len(_dvs))):
        ax.annotate(short_variant(r['Variant']), xy=(r['Year AUC (macro OvR)'], r['Depression AUC (forward, pooled)']),
                    xytext=(0.52, yl), textcoords='data', fontsize=6.5, va='center',
                    arrowprops=dict(arrowstyle='-', lw=0.4, color='#9E9E9E', shrinkA=0, shrinkB=3))
    ax.legend(handles=[Patch(color='#0072B2', label='All / all but Wi-Fi'), Patch(color='#009E73', label='Single modality'),
                       Patch(color='#D55E00', label='Missingness / availability manipulations')], frameon=False, fontsize=7, loc='upper left')
    plt.tight_layout()
    save_fig(fig, 'Fig2_year_vs_depression_auc', 'Collection-year identifiability versus depression-risk discrimination, overall and across feature subsets.', main=True)
    plt.show()

# ---- Figure 3: performance across protocols, years and models --------------
VT = VIEW_TABLES['main']
fig, axes = plt.subplots(1, 3, figsize=(11.5, 4.0), sharey=True,
                         gridspec_kw={'width_ratios': [len(views_of('main', PROTO_FWD)) + 1, len(views_of('main', PROTO_LOYO)) + 1, 1.3]})
ms = model_order(VT['Model'].unique())
for ax, proto, title in zip(axes, (PROTO_FWD, PROTO_LOYO, PROTO_PD),
                            ('A  Forward-chaining (primary)', 'B  Leave-one-year-out', 'C  Participant-disjoint')):
    vs = views_of('main', proto)
    cy = common_years('main', proto)
    labels_ = ([v['view'].replace('Forward \u2192 ', '\u2192').replace('Leave-out ', '') for v in vs] +
               ([f'Pooled\n{_yr_range(cy)}'] if proto != PROTO_PD else []))
    for k, m in enumerate(ms):
        off = (k - (len(ms) - 1) / 2) * 0.09
        for j, v in enumerate(vs):
            r = VT[(VT['split'] == v['split']) & (VT['Model'] == m) & (VT['Protocol'] == proto)]
            if len(r):
                r = r.iloc[0]
                ax.errorbar(j + off, r['AUC'], yerr=[[r['AUC'] - r['AUC_lo']], [r['AUC_hi'] - r['AUC']]], fmt='o', ms=3.5,
                            color=MODEL_COLORS.get(m), lw=0.9, capsize=1.5)
        if proto != PROTO_PD:
            pr = pooled('main', proto, m, years=cy)
            if pr is not None:
                ax.errorbar(len(vs) + off, pr['point'], yerr=[[pr['point'] - pr['lo']], [pr['hi'] - pr['point']]], fmt='D', ms=4,
                            color=MODEL_COLORS.get(m), lw=1.2, capsize=2)
    if proto != PROTO_PD:
        ax.axvline(len(vs) - 0.5, color='grey', lw=0.6)
    ax.axhline(0.5, color='black', ls='--', lw=1)
    ax.set_xticks(range(len(labels_))); ax.set_xticklabels(labels_)
    ax.set_title(title, loc='left')
axes[0].set_ylabel('ROC AUC (95% CI: participants x seeds)')
fig.legend(handles=[Patch(color=MODEL_COLORS[m], label=m) for m in ms], loc='lower center', ncol=len(ms), frameon=False,
           bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()
save_fig(fig, 'Fig3_performance', 'Depression-risk ROC AUC by protocol, test year and model. Pooled estimates average the test years shared by every model (GroupDRO/IRM need two training years, so the first forward year is not pooled).', main=True)
plt.show()

# ---- Figure 4: calibration and PPV at deployment prevalence ----------------
fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.2))
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', lw=1)
_cyf = common_years('main', PROTO_FWD)
fwd_splits = [v['split'] for v in views_of('main', PROTO_FWD) if v['test_year'] in _cyf]
for m in model_order(models_in('main')):
    if m == 'Dummy':
        continue
    d = pd.concat([df for (a, mm, s, seed), df in PRED.items() if a == 'main' and mm == m and s in fwd_splits])
    q = pd.qcut(d['proba'].rank(method='first'), 10, labels=False)
    g = d.groupby(q).agg(p=('proba', 'mean'), o=('y_true', 'mean'))
    ax.plot(g['p'], g['o'], 'o-', ms=3, lw=1.2, color=MODEL_COLORS[m], label=m)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel('Mean predicted probability (decile)'); ax.set_ylabel('Observed positive rate')
ax.set_title(f'A  Calibration, forward-chaining {_yr_range(_cyf)} (seeds pooled)', loc='left')
ax.legend(frameon=False, fontsize=7)
ax = axes[1]
grid = np.linspace(0.01, 0.5, 100)
_band_models = [r for r in (KEY_NUMBERS.get('forward_best_ppv_at_10pct', {}).get('model'), 'XGBoost') if r]
for m in model_order(models_in('main')):
    if m == 'Dummy':
        continue
    op = operating_point('main', PROTO_FWD, m, years=_cyf)
    if op is None:
        continue
    pv = ppv_fn(op['sens'], op['spec'], grid)
    band = np.array([ci(ppv_fn(op['sens_b'], op['spec_b'], g_)) for g_ in grid])
    ax.plot(100 * grid, 100 * pv, color=MODEL_COLORS[m], lw=1.4, label=m)
    if m in _band_models:
        ax.fill_between(100 * grid, 100 * band[:, 0], 100 * band[:, 1], color=MODEL_COLORS[m], alpha=0.15, lw=0)
ax.plot(100 * grid, 100 * grid, color='grey', ls=':', lw=1, label='No-skill (PPV = prevalence)')
for p_ in (5, 10):
    ax.axvline(p_, color='black', ls='--', lw=0.8)
_b10 = KEY_NUMBERS.get('forward_best_ppv_at_10pct')
if _b10:
    ax.text(11, 3, f"best PPV at 10%: {100*_b10['ppv']:.1f}%\n({_b10['false_alerts_per_true_alert']:.1f} false alerts per true alert)", fontsize=7)
ax.set_xlabel('Prevalence in deployed population (%)'); ax.set_ylabel('Positive predictive value (%)')
ax.set_title('B  Implied PPV at the validation-selected threshold', loc='left')
ax.set_xlim(0, 50); ax.set_ylim(0, 70)
ax.legend(frameon=False, fontsize=7, ncol=2, loc='upper left', bbox_to_anchor=(0.21, 1.0))   # clear of the 5%/10% lines
plt.tight_layout()
save_fig(fig, 'Fig4_calibration_ppv', 'Calibration of forward-chaining predictions and implied positive predictive value across deployment prevalence (bands: 95% CI for XGBoost and the best model at 10%).', main=True)
plt.show()

# ---- Figure 5: explanation stability ----------------------------------------
if len(STAB_PAIRS):
    SP = STAB_PAIRS[STAB_PAIRS['analysis'] == 'main']
    SS = STAB_SUMMARY[STAB_SUMMARY['analysis'] == 'main'].set_index(['code', 'level'])
    fig, axes = plt.subplots(1, 3, figsize=(12.0, 4.0), gridspec_kw={'width_ratios': [1.1, 1.2, 1.3]})
    ax = axes[0]
    codes = [c for c in 'ABCD' if (c, 'cluster') in SS.index]
    cc = {'A': '#009E73', 'B': '#56B4E9', 'C': '#E69F00', 'D': '#D55E00'}
    rng_ = np.random.default_rng(0)
    for j, c in enumerate(codes):
        v = SP[(SP['comparison'] == c) & (SP['level'] == 'cluster')]['jaccard'].dropna().to_numpy()
        ax.scatter(j + rng_.uniform(-0.18, 0.18, len(v)), v, s=5, alpha=0.25, color=cc[c], lw=0)
        r = SS.loc[(c, 'cluster')]
        ax.errorbar(j, r['jaccard'], yerr=[[r['jaccard'] - r['jaccard_lo']], [r['jaccard_hi'] - r['jaccard']]], fmt='o', color='black', ms=5, capsize=3)
    ax.set_xticks(range(len(codes))); ax.set_xticklabels([CMP_LABELS[c].replace(' (', '\n(').replace(', ', ',\n') for c in codes], fontsize=7)
    ax.set_ylim(-0.02, 1.02); ax.set_ylabel(f'Jaccard@{TOP_K} of top feature clusters')
    ax.set_title(f'A  Top-{TOP_K} overlap of feature clusters', loc='left', fontsize=9)
    ax = axes[1]
    lv = ['cluster', 'construct', 'modality']
    for c in codes:
        y_ = [SS.loc[(c, l), 'spearman_top'] if (c, l) in SS.index else np.nan for l in lv]
        lo_ = [SS.loc[(c, l), 'spearman_top_lo'] if (c, l) in SS.index else np.nan for l in lv]
        hi_ = [SS.loc[(c, l), 'spearman_top_hi'] if (c, l) in SS.index else np.nan for l in lv]
        off = (codes.index(c) - 1.5) * 0.08
        ax.errorbar(np.arange(3) + off, y_, yerr=[np.subtract(y_, lo_), np.subtract(hi_, y_)], fmt='o-', color=cc[c], ms=4,
                    capsize=2, lw=1.2, label=CMP_LABELS[c])
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xticks(range(3)); ax.set_xticklabels(['Feature cluster', 'Construct', 'Modality'])
    ax.set_ylabel(f'Spearman within each ranking\'s top-{TOP_UNION}')
    ax.set_title('B  Rank agreement by aggregation level', loc='left', fontsize=9)
    ax.legend(frameon=False, fontsize=6.5, loc='lower right')
    ax = axes[2]
    if len(modality_shares):
        msum = modality_shares.groupby(['modality', 'year'])['share'].agg(['mean', 'std']).reset_index()
        mods_ = (msum.groupby('modality')['mean'].mean().sort_values(ascending=False).index.tolist())
        yrs_ = sorted(msum['year'].unique())
        for k, y in enumerate(yrs_):
            s_ = msum[msum['year'] == y].set_index('modality').reindex(mods_)
            ax.errorbar(np.arange(len(mods_)) + (k - (len(yrs_) - 1) / 2) * 0.15, 100 * s_['mean'], yerr=100 * s_['std'].fillna(0),
                        fmt='o', ms=4, capsize=2, color=PALETTE[k % len(PALETTE)], label=f'Held-out {y}')
        ax.set_xticks(range(len(mods_))); ax.set_xticklabels(mods_, rotation=30, ha='right')
        ax.set_ylabel('Share of |SHAP| (%), mean \u00b1 SD over seeds')
        ax.legend(frameon=False, fontsize=7)
    ax.set_title('C  Modality share of |SHAP| by held-out year', loc='left', fontsize=9)
    plt.tight_layout()
    save_fig(fig, 'Fig5_explanation_stability', 'Explanation stability: within-year, seed-refit, same-model-across-year and cross-year agreement of SHAP rankings at three levels of aggregation.', main=True)
    plt.show()

## 16. Main tables and supplementary material

In [ ]:
save_table(cohort_table, 'Table1_cohorts', 'Cohort composition, label source and prevalence by collection year.', main=True)
_t2 = POOLED_TABLES['main'][['Protocol', 'Model', 'Years', 'AUC (95% CI)', 'BA', 'BA_lo', 'BA_hi',
                             'Years (all available)', 'AUC all available years (95% CI)']].round(4)
save_table(_t2, 'Table2_pooled_performance', 'Pooled ROC AUC (95% CI from the two-level bootstrap) by protocol and model: '
           'primary columns pool the test years shared by every model; the last two columns pool all years available to each model.', main=True)

S = []   # (df, name, caption)
S.append((label_source_table, 'S01_label_sources', 'Weekly label construction and survey columns per year.'))
S.append((pid_linkage_table, 'S02_pid_linkage', 'Participant-ID overlap across years in the released data (IDs do not link).'))
S.append((split_composition, 'S03_split_composition', 'Composition of every evaluation split (views and computed splits).'))
_cfg = []
for m in MODELS_ALL:
    _cfg.append({'Model': m, 'Default configuration': json.dumps(DEFAULT_PARAMS[m]),
                 'Optuna search space': SEARCH_SPACE_TEXT.get(m, SEARCH_SPACE_TEXT['MLP'] if m.startswith('MLP') else 'none'),
                 'Trials per outer split': HPO_TRIALS.get(m, 0)})
_cfg.append({'Model': 'All', 'Default configuration': f'seeds={SEEDS}; inner validation={INNER_VAL_SIZE:.0%} of training participants',
             'Optuna search space': 'TPE sampler; objective = inner-validation ROC AUC; default enqueued as trial 0; median pruning for neural models',
             'Trials per outer split': ''})
S.append((pd.DataFrame(_cfg), 'S04_model_configurations', 'Model configurations and hyperparameter search spaces.'))
_hp = []
for (an, m, sp_), v in HPO.items():
    _hp.append({'Analysis': an, 'Model': m, 'Split': sp_, 'Default val AUC': v['default_value'], 'Tuned val AUC': v['best_value'],
                'Trials': v['n_trials'], 'Pruned': v['n_pruned'], 'Selected parameters': json.dumps(v['best_params'], default=float)})
HPO_TABLE = pd.DataFrame(_hp)
S.append((HPO_TABLE, 'S05_hpo_selected_parameters', 'Optuna results per analysis, model and outer split (inner validation only).'))
_m = ['roc_auc', 'pr_auc', 'balanced_accuracy', 'f1', 'sensitivity', 'specificity', 'precision', 'brier', 'pred_positive_rate']
_vm = pd.DataFrame(VIEWS)[['analysis', 'protocol', 'view', 'split']]
_full = RESULTS.merge(_vm, on=['analysis', 'split'])
_g = _full.groupby(['analysis', 'protocol', 'view', 'model'])[_m]
FULL_RESULTS = pd.DataFrame({k: [f'{a:.3f} \u00b1 {b:.3f}' for a, b in zip(_g.mean()[k], _g.std()[k])] for k in _m}, index=_g.mean().index).reset_index()
S.append((FULL_RESULTS, 'S06_full_results', 'All metrics (mean \u00b1 SD over seeds) per analysis, view and model.'))
S.append((pd.concat(VIEW_TABLES.values(), ignore_index=True).round(4), 'S07_bootstrap_by_view',
          'Two-level bootstrap 95% CIs per view and model, with seed, participant and total SDs.'))
S.append((pd.concat([c for c in COMPARISONS.values()], ignore_index=True).round(4), 'S08_paired_comparisons_pooled',
          'Pooled paired comparisons (Holm-adjusted within family; equivalence margin \u00b10.02 AUC).'))
S.append((pd.concat([c for c in PER_VIEW_COMPARISONS.values()], ignore_index=True).round(4), 'S09_paired_comparisons_by_view',
          'Per-view paired comparisons against XGBoost (Holm-adjusted within view).'))
_cal = calibration_summary.copy()
_cal.columns = [f'{a}_{b}' for a, b in _cal.columns]          # single header row (slope_mean, slope_std, ...)
S.append((_cal.reset_index(), 'S10_calibration', 'Calibration slope, calibration-in-the-large, ECE and Brier (mean, SD over views and seeds).'))
S.append((PPV_TABLE.round(4), 'S11_ppv_by_prevalence', 'PPV, NPV, alerts per 1,000 and false alerts per true alert at assumed prevalences.'))
S.append((domain_year_table.round(4), 'S12_year_classifier_variants', 'Collection-year classifier AUC across feature subsets, eras and imputation schemes.'))
S.append((domain_variant_table.round(4), 'S13_year_vs_depression_variants', 'Year AUC and forward-chaining depression AUC on the same feature subsets.'))
S.append(((100 * year_shap_shares).round(2).reset_index() if len(year_shap_shares) else year_shap_shares,
          'S14_year_classifier_shap_shares', 'Share (%) of year-classifier |SHAP| by modality and feature type.'))
S.append((era_table.round(4), 'S15_era_transfer', 'Depression AUC within and across pre-COVID/COVID eras (XGBoost, simplified pipeline).'))
S.append((STAB_SUMMARY.round(4), 'S16_explanation_stability', 'Explanation-stability summary by comparison type and aggregation level.'))
S.append((STAB_PERM.round(4), 'S17_same_model_year_permutation', 'Same-model year-partition permutation test (lower similarity than random partitions => year-dependent explanations).'))
S.append((kendall_table.round(4), 'S18_kendall_w', "Kendall's W across LOYO models and across seeds within year."))
S.append((top_clusters.round(5), 'S19_top_clusters_latest_year', 'Top SHAP feature clusters on the latest forward split (mean over seeds, top-20 frequency).'))
S.append((top_constructs.round(5), 'S20_top_constructs_latest_year', 'Top behavioural constructs on the latest forward split.'))
if len(modality_shares):
    S.append((modality_shares.groupby(['modality', 'year'])['share'].agg(['mean', 'std']).reset_index().round(4),
              'S21_modality_shares', 'Modality share of |SHAP| by held-out year (mean, SD over seeds).'))
S.append((sens_summary, 'S22_sensitivity_2018_summary', 'Do conclusions change when 2018 (PANAS-derived labels) is excluded? Includes BDI-II consistent-outcome results.'))
if 'ex2018' in ANALYSES:
    S.append((sens_model_table.round(4), 'S23_sensitivity_2018_by_model', 'Main vs 2018-excluded pooled AUC per model on identical test years (paired).'))
if 'bdi' in ANALYSES:
    S.append((POOLED_TABLES['bdi'].round(4), 'S24_bdi_endterm_pooled', 'End-of-term BDI-II (> 13) outcome: pooled AUC by protocol and model.'))
    S.append((COMPARISONS['bdi'].round(4), 'S25_bdi_endterm_comparisons', 'End-of-term BDI-II outcome: paired comparisons.'))
S.append((norm_ablation_table.round(4), 'S26_norm_ablation', '`_norm` ablation (tabular models): paired differences, excluded minus included.'))
if len(perm_results):
    S.append((perm_summary.round(4), 'S27_permutation_test', 'User-level label-permutation negative control (tuned XGBoost).'))
_rq = RESULTS.groupby(['analysis', 'model']).agg(runs=('seed', 'size'), collapsed=('collapsed', 'sum'),
                                                 val_auc_below_052=('val_auc', lambda s: int((s < 0.52).sum()))).reset_index()
S.append((_rq, 'S28_run_quality', 'Collapsed runs and runs with inner-validation AUC < 0.52.'))
S.append(((100 * daily_missingness).round(1).reset_index().rename(columns={'index': 'Modality'}), 'S29_daily_missingness',
          'Missing daily observations (%) by modality and year.'))
_plat = (ml_table.drop_duplicates(['dataset', 'pid']).groupby(['year', 'platform']).size().unstack(fill_value=0))
S.append((_plat.reset_index(), 'S30_platform_by_year', 'Device platform of participants by year.'))
_ps = RESULTS[['analysis', 'model', 'split', 'seed', 'roc_auc', 'pr_auc', 'balanced_accuracy', 'sensitivity', 'specificity',
               'decision_threshold', 'val_auc', 'best_epoch', 'collapsed']]
S.append((_ps.round(4), 'S31_per_seed_results', 'Per-seed results for every analysis, split and model.'))
S.append((pd.DataFrame(STAGE_TIMES), 'S32_runtime', 'Runtime per pipeline stage (this session).'))
if RUN_BDI_ENDTERM:
    S.append((bdi_source_table, 'S33_bdi_label_source', 'End-of-term BDI-II label construction per year (strict: no fallback definition).'))
_pooled_keep = _pooled_miss[_cand] <= MAX_MISSING_FRAC
_cf = []
for _sp, _S in SPLITS.items():
    _k = split_channel_mask(_sp)
    _cf.append({'Split': _sp, 'Train years': ', '.join(map(str, _S['train_years'])),
                'Channels kept (training-only rule)': int(_k.sum()),
                'Kept only by training-only rule': int((_k & ~_pooled_keep).sum()),
                'Kept only by pooled all-years rule': int((~_k & _pooled_keep).sum())})
S.append((pd.DataFrame(_cf), 'S34_channel_filter_by_split',
          'Raw channels kept by the training-only missingness rule for each split versus a pooled all-years rule.'))
if len(PROTOCOL_LADDER):
    S.append((PROTOCOL_LADDER.round(4), 'S35_loyo_vs_forward_same_test_year',
              'Paired comparison on the same test participants and bootstrap resamples: leave-one-year-out (later year(s) '
              'added to training) minus forward-chaining. Differences combine later-year data and a larger training set; '
              'Holm-adjusted across models and years.'))
for df_, n_, c_ in S:
    save_table(df_, n_, c_)

# ---- Supplementary figures ---------------------------------------------------
fvs = views_of('main', PROTO_FWD)
ms_cm = [m for m in model_order(models_in('main'))]
fig, axes = plt.subplots(len(fvs), len(ms_cm), figsize=(1.6 * len(ms_cm), 1.7 * len(fvs)), squeeze=False)
for i, v in enumerate(fvs):
    for j, m in enumerate(ms_cm):
        ax = axes[i][j]
        ds_ = [df for (a, mm, s, seed), df in PRED.items() if a == 'main' and mm == m and s == v['split']]
        ax.axis('off')
        if not ds_:
            continue
        d = pd.concat(ds_)
        cm_ = confusion_matrix(d['y_true'], d['pred'], labels=[0, 1]) / len(ds_)
        ax.imshow(cm_ / cm_.sum(), cmap='Blues', vmin=0, vmax=0.6)
        for r_ in range(2):
            for c_ in range(2):
                ax.text(c_, r_, f'{cm_[r_, c_]:.0f}', ha='center', va='center', fontsize=7)
        if i == 0:
            ax.set_title(m, fontsize=7)
        if j == 0:
            ax.text(-0.9, 0.5, v['view'], rotation=90, va='center', ha='center', fontsize=7)
plt.tight_layout()
save_fig(fig, 'S_Fig01_confusion_matrices', 'Confusion matrices (mean counts over seeds; rows true 0/1, columns predicted 0/1) for forward-chaining views.')
plt.show()

if len(HPO_TABLE.dropna(subset=['Default val AUC', 'Tuned val AUC'])):
    fig, ax = plt.subplots(figsize=(4.6, 4.2))
    h = HPO_TABLE.dropna(subset=['Default val AUC', 'Tuned val AUC'])
    for m in h['Model'].unique():
        hh = h[h['Model'] == m]
        ax.scatter(hh['Default val AUC'], hh['Tuned val AUC'], s=14, color=MODEL_COLORS.get(m), label=m)
    lim = [min(h['Default val AUC'].min(), h['Tuned val AUC'].min()) - 0.02, max(h['Default val AUC'].max(), h['Tuned val AUC'].max()) + 0.02]
    ax.plot(lim, lim, 'k:', lw=1); ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('Inner-validation AUC, default configuration'); ax.set_ylabel('Inner-validation AUC, Optuna best')
    ax.legend(frameon=False, fontsize=7)
    save_fig(fig, 'S_Fig02_hpo_default_vs_tuned', 'Inner-validation AUC of the default configuration (trial 0) versus the Optuna-selected configuration.')
    plt.show()

def _forest(rows_, title, name, caption, xlabel='ROC AUC'):
    if not rows_:
        return
    fig, ax = plt.subplots(figsize=(6.4, 0.32 * len(rows_) + 1.2))
    for i, (lab, p_, lo, hi, col) in enumerate(rows_):
        ax.errorbar(p_, i, xerr=[[p_ - lo], [hi - p_]], fmt='o', color=col, ms=4, capsize=2)
    ax.set_yticks(range(len(rows_))); ax.set_yticklabels([r[0] for r in rows_], fontsize=7)
    ax.axvline(0.5 if xlabel == 'ROC AUC' else 0.0, color='black', ls='--', lw=1)
    ax.invert_yaxis(); ax.set_xlabel(xlabel); ax.set_title(title, loc='left')
    ax.grid(axis='y', visible=False)
    save_fig(fig, name, caption)
    plt.show()

if 'ex2018' in ANALYSES:
    _rows = []
    for proto, yrs_ in ((PROTO_FWD, [v['test_year'] for v in views_of('ex2018', PROTO_FWD)]),
                        (PROTO_LOYO, [v['test_year'] for v in views_of('ex2018', PROTO_LOYO)])):
        for m in model_order(models_in('ex2018')):
            ym = sorted(set(model_years('main', proto, m)) & set(model_years('ex2018', proto, m)) & set(yrs_))
            for an, col in (('main', '#0072B2'), ('ex2018', '#D55E00')):
                r = pooled(an, proto, m, years=ym) if ym else None
                if r is not None:
                    _rows.append((f'{proto[:7]} {_yr_range(ym)} | {m} | {"main" if an == "main" else "excl. 2018"}',
                                  r['point'], r['lo'], r['hi'], col))
    _forest(_rows, 'Sensitivity: 2018 excluded (orange) vs main (blue), same test years', 'S_Fig03_sensitivity_ex2018',
            'Pooled AUC on identical test years with and without 2018 in training.')
if 'bdi' in ANALYSES:
    _rows = [(f'{r["Protocol"][:7]} | {r["Model"]}', r['AUC'], r['AUC_lo'], r['AUC_hi'], MODEL_COLORS.get(r['Model'], 'k'))
             for _, r in POOLED_TABLES['bdi'].iterrows()]
    _forest(_rows, 'Consistent outcome: end-of-term BDI-II > 13', 'S_Fig04_bdi_endterm', 'Pooled AUC for the end-of-term BDI-II outcome.')
if len(norm_ablation_table):
    _rows = [(f'{r["Protocol"][:7]} | {r["Model"]}', r['Delta (excl - incl)'], r['CI_lo'], r['CI_hi'], MODEL_COLORS.get(r['Model'], 'k'))
             for _, r in norm_ablation_table.iterrows()]
    _forest(_rows, '`_norm` ablation: AUC(excluded) - AUC(included)', 'S_Fig05_norm_ablation', 'Paired AUC differences when `_norm` features are removed.',
            xlabel='Delta ROC AUC')
if len(era_table):
    _rows = [(f'{r["Era contrast"]} | {r["Train \u2192 test"]}', r['AUC'], r['AUC_lo'], r['AUC_hi'], '#0072B2') for _, r in era_table.iterrows()]
    _forest(_rows, 'Depression transfer within and across COVID eras', 'S_Fig06_era_transfer', 'XGBoost AUC within vs across pre-COVID/COVID eras.')
if len(year_shap_shares):
    fig, ax = plt.subplots(figsize=(6.4, 3.4))
    ys = 100 * year_shap_shares
    bottom = np.zeros(len(ys))
    for k, col in enumerate(ys.columns):
        ax.barh(ys.index, ys[col], left=bottom, color=PALETTE[k % len(PALETTE)], label=col)
        bottom += ys[col].to_numpy()
    ax.set_xlabel('Share of year-classifier |SHAP| (%)'); ax.legend(frameon=False, fontsize=7)
    ax.set_title('What makes collection year identifiable?', loc='left')
    save_fig(fig, 'S_Fig07_year_classifier_shap', 'Year-classifier attribution by modality and feature type (value, missingness, self-deviation, availability).')
    plt.show()
if len(perm_results):
    fig, axes = plt.subplots(1, perm_results['split'].nunique(), figsize=(4.2 * perm_results['split'].nunique(), 3.0), squeeze=False)
    for ax, (s_, g_) in zip(axes[0], perm_results.groupby('split')):
        ax.hist(g_['roc_auc'], bins=12, color='#9E9E9E')
        o = perm_summary.set_index('split').loc[s_, 'observed']
        ax.axvline(o, color='#D55E00', lw=2); ax.set_title(s_.split('|')[1], fontsize=8); ax.set_xlabel('Permuted-label AUC')
    save_fig(fig, 'S_Fig08_permutation_null', 'Label-permutation null distributions with the observed seed-mean AUC.')
    plt.show()
fig, ax = plt.subplots(figsize=(5.2, 3.2))
_hm = daily_missingness
ax.imshow(_hm.to_numpy(dtype=float), cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(_hm.shape[1])); ax.set_xticklabels(_hm.columns); ax.set_yticks(range(_hm.shape[0])); ax.set_yticklabels(_hm.index)
for i in range(_hm.shape[0]):
    for j in range(_hm.shape[1]):
        ax.text(j, i, f'{100*_hm.iloc[i, j]:.0f}%', ha='center', va='center', fontsize=7)
ax.grid(visible=False); ax.set_title('Missing daily observations by modality and year', loc='left')
save_fig(fig, 'S_Fig09_missingness_heatmap', 'Missing daily observations by modality and collection year.')
plt.show()

## 17. Export: key numbers, artefact index, manifest, zip

In [ ]:
KEY_NUMBERS['n_windows'] = int(len(meta)); KEY_NUMBERS['n_participant_years'] = int(meta.groupby(['dataset', 'pid']).ngroups)
KEY_NUMBERS['seeds'] = len(SEEDS); KEY_NUMBERS['bootstrap_replicates'] = N_BOOT
(OUTPUT_ROOT / 'key_numbers.json').write_text(json.dumps(KEY_NUMBERS, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))
pd.DataFrame(ARTEFACTS).to_csv(OUTPUT_ROOT / 'artefact_index.csv', index=False)
manifest = dict(run_name=RUN_NAME, config_hash=CONFIG_HASH, config=CONFIG, checkpoints=dict(CK_STATS),
                checkpoint_version_mismatch=CK_VERSION_MISMATCH, checkpoint_load_errors=CK_LOAD_ERRORS, fingerprint_fallbacks=sorted(set(FP_FALLBACKS)),
                data_fingerprint=DATA_FP, preprocessing_fingerprint=PREP_FP, library_versions=LIB_VERSIONS, exported=datetime.datetime.now().isoformat(timespec='seconds'),
                analyses=list(ANALYSES), n_splits=len(SPLITS), python=platform.python_version(), torch=torch.__version__,
                xgboost=xgb.__version__, optuna=optuna.__version__, sklearn=__import__('sklearn').__version__,
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
(OUTPUT_ROOT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
(OUTPUT_ROOT / 'README.md').write_text(f"""# GLOBEM temporal-generalisation analysis outputs ({RUN_NAME})

* `main_figures/` - Figures 1-5 (PNG 300 dpi + PDF).
* `main_tables/` - Table 1 (cohorts) and Table 2 (pooled performance).
* `supplementary/` - every other table (S01-S35) and figure (S_Fig01-09); `artefact_index.csv` lists captions.
* `key_numbers.json` - numbers quoted in the manuscript text (regenerated on every run).
* `checkpoints/` - resumable intermediate results keyed by code/settings/input fingerprints (contain participant-level predictions: never share; safe to delete after the run).

Inference: two-level bootstrap ({N_BOOT} replicates; participants resampled with replacement and a training seed drawn
per replicate), {len(SEEDS)} seeds per model, Holm adjustment within comparison families, equivalence margin +/-{EQUIV_MARGIN} AUC.
Hyperparameters: Optuna on the participant-disjoint inner validation set only; outer test years untouched.
""")
_zip = OUTPUT_ROOT.parent / f'GLOBEM_outputs_{RUN_NAME}_{datetime.datetime.now():%Y%m%d_%H%M}.zip'
with zipfile.ZipFile(_zip, 'w', zipfile.ZIP_DEFLATED) as zf:   # checkpoints are excluded (large, resumable state)
    for p in OUTPUT_ROOT.rglob('*'):
        if p.is_file() and 'checkpoints' not in p.relative_to(OUTPUT_ROOT).parts:
            zf.write(p, p.relative_to(OUTPUT_ROOT))
print('Key numbers:', json.dumps(KEY_NUMBERS, indent=1, default=str)[:3000])
print('\nArtefacts:', len(ARTEFACTS), '| bundle:', _zip)
print('Checkpoints this session:', CK_STATS, '| fingerprint fallbacks:', sorted(set(FP_FALLBACKS)),
      '| library-version mismatches:', len(CK_VERSION_MISMATCH), '| unreadable checkpoints:', len(CK_LOAD_ERRORS))
display(pd.DataFrame(STAGE_TIMES))